# Select core emotion features

Run the setup cell first. Review model, SAE, data paths, and run settings before starting. See the repository README for setup and pipeline order.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
import os

CACHE_ROOT = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_ROOT}/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_ROOT}/transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{CACHE_ROOT}/hub"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT

HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Authentication is read from HF_TOKEN or the local Hugging Face login.
# Authentication is read from HF_TOKEN or the local Hugging Face login.

In [ ]:
from sae_emotion.emotion_entropy_utils import run_entropy_classification_pipeline


INPUT_PATH = "data/google-gemma-2-2b-feature-act/core_emotion_features_filled_logits_signed.csv"

OUTPUT_PATH = "data/google-gemma-2-2b-feature-act/core_emotion_features_filled_logits_signed_entropy_test.csv"

PLOT_DIR = "data/google-gemma-2-2b-feature-act/visuals"


df, threshold = run_entropy_classification_pipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    entropy_threshold=None,  # Set to 0.7204 if you want fixed threshold
    plot_dir=PLOT_DIR,
    show_plots=True,
)

print("Final entropy threshold:", threshold)

### cumulative method

In [ ]:
# # -----------------------
# # Strongest emotion-specific features
# # based on cumulative positive contribution
# # -----------------------

# # use positive-only max logit
# vals = df_specific["max_positive_logit"].values

# # remove zero rows
# vals = vals[vals > 0]

# # sort descending
# vals_sorted = np.sort(vals)[::-1]

# # cumulative contribution
# cum = np.cumsum(vals_sorted)
# cum = cum / cum[-1]

# # keep features accounting for 90% total positive effect
# TARGET_CUM_RATIO = 0.9

# idx = np.argmax(cum >= TARGET_CUM_RATIO)

# threshold = vals_sorted[idx]

# print("Cumulative contribution threshold:", threshold)

# # -----------------------
# # Select features
# # -----------------------
# df_specific_top = df_specific[
#     df_specific["max_positive_logit"] >= threshold
# ].copy()

# print("Selected rows:", len(df_specific_top))

# print(
#     df_specific_top[
#         [
#             "layer_idx",
#             "feature_idx",
#             "max_positive_logit",
#             "normalized_entropy",
#         ]
#     ].head()
# )

# # -----------------------
# # Select strong features
# # -----------------------
# df_specific_top = df_specific[
#     df_specific["max_positive_logit"] >= threshold
# ].copy()

# # -----------------------
# # Add affected_emotions column
# # -----------------------
# logit_cols = [f"logit_{e}" for e in EMOTIONS]

# def get_affected_emotions(row):
#     vals = []

#     for e in EMOTIONS:
#         # ignore negative effects
#         v = row[f"logit_{e}"]

#         # only positive logits crossing threshold count
#         if v >= threshold:
#             vals.append((e, v))

#     vals = sorted(vals, key=lambda x: x[1], reverse=True)

#     return ",".join([e for e, _ in vals])

# df_specific_top["affected_emotions"] = df_specific_top.apply(
#     get_affected_emotions,
#     axis=1
# )

# df_specific_top["num_affected_emotions"] = (
#     df_specific_top["affected_emotions"]
#     .apply(lambda x: 0 if x == "" else len(x.split(",")))
# )

# # -----------------------
# # Print summary
# # -----------------------
# print("Target cumulative ratio:", TARGET_CUM_RATIO)
# print("Threshold:", threshold)
# print("Selected rows:", len(df_specific_top))

# print(
#     df_specific_top[
#         [
#             "layer_idx",
#             "feature_idx",
#             "max_positive_logit",
#             "normalized_entropy",
#             "affected_emotions",
#             "num_affected_emotions",
#         ]
#     ].head()
# )

# # -----------------------
# # Save
# # -----------------------
# df_specific_top.to_csv(
#     "data/google-gemma-2-2b-feature-act/core_emotion_features_filled_logits_signed_entropy_top_cumulative0.9.csv",
#     index=False
# )

### top-k method

In [ ]:
# import os
# import gc
# import numpy as np
# import pandas as pd
# import torch
# from tqdm import tqdm

# from logit_ablation_utils import (
#     load_llm,
#     SAECache,
#     build_emotion_token_map,
#     build_examples_by_emotion,
#     model_safe_name,
# )

# # ============================================================
# # Config
# # ============================================================

# MODEL_NAME = "google/gemma-2-2b"
# SAFE_NAME = model_safe_name(MODEL_NAME)

# FEATURE_CSV_PATH = "data/google-gemma-2-2b-feature-act/core_emotion_features_filled_logits_signed_entropy.csv"
# DATASET_CSV_PATH = f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"

# OUT_PATH = "data/google-gemma-2-2b-feature-act/topk_joint_ablation_logit_drop.csv"

# EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]
# TOP_K_LIST = [1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20]

# TEXT_COL = "input_text"
# LABEL_COL = "emotion"

# CACHE_DIR = os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub"))

# SAMPLE_SIZE_PER_EMOTION = 300
# BATCH_SIZE = 128
# RANDOM_STATE = 42
# APPLY_PROMPT_TEMPLATE = False

# DEVICE = "cuda"
# DTYPE = torch.bfloat16

# USE_ABS_RANKING = False
# SAVE_EVERY_ROW = True


# # ============================================================
# # Helpers
# # ============================================================

# def cleanup_memory():
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()


# def safe_save_csv(df: pd.DataFrame, path: str):
#     os.makedirs(os.path.dirname(path), exist_ok=True)
#     tmp_path = path + ".tmp"
#     df.to_csv(tmp_path, index=False)
#     os.replace(tmp_path, path)


# def zero_ablate_features(
#     sae,
#     source_sae_features: torch.Tensor,
#     feature_indices,
#     residual: torch.Tensor,
# ) -> torch.Tensor:
#     """
#     Jointly zero-ablate multiple SAE features at one layer.
#     """
#     modified_sae_features = source_sae_features.clone()
#     feature_indices = [int(x) for x in feature_indices]

#     modified_sae_features[:, feature_indices] = 0.0

#     modified_hidden = sae.decode(modified_sae_features)
#     return modified_hidden + residual


# def compute_topk_joint_logit_drop(
#     llm,
#     sae,
#     examples,
#     layer_idx,
#     feature_indices,
#     target_token,
#     batch_size=128,
# ):
#     """
#     Ablate a set of SAE features together and measure target emotion logit drop.

#     signed_drop = clean_logit - ablated_logit

#     Positive signed_drop means removing these features lowers the target emotion logit.
#     """
#     all_signed_drops = []
#     all_abs_drops = []
#     all_clean_logits = []
#     all_ablated_logits = []

#     for start in range(0, len(examples), batch_size):
#         batch_examples = examples[start:start + batch_size]

#         with torch.no_grad():
#             with llm.trace(batch_examples):
#                 source_hidden = (
#                     llm.model.layers[int(layer_idx)]
#                     .output[:, -1, :]
#                     .save()
#                 )

#                 clean_logits = (
#                     llm.output.logits[:, -1, int(target_token)]
#                     .save()
#                 )

#         source_sae_features = sae.encode(source_hidden)
#         source_hidden_decoded = sae.decode(source_sae_features)
#         residual = source_hidden - source_hidden_decoded

#         with torch.no_grad():
#             with llm.trace(batch_examples):
#                 llm.model.layers[int(layer_idx)].output[:, -1, :] = zero_ablate_features(
#                     sae=sae,
#                     source_sae_features=source_sae_features,
#                     feature_indices=feature_indices,
#                     residual=residual,
#                 )

#                 ablated_logits = (
#                     llm.output.logits[:, -1, int(target_token)]
#                     .save()
#                 )

#         signed_drop = clean_logits - ablated_logits
#         abs_drop = signed_drop.abs()

#         all_signed_drops.append(signed_drop.detach().float().cpu())
#         all_abs_drops.append(abs_drop.detach().float().cpu())
#         all_clean_logits.append(clean_logits.detach().float().cpu())
#         all_ablated_logits.append(ablated_logits.detach().float().cpu())

#         del source_hidden
#         del clean_logits
#         del source_sae_features
#         del source_hidden_decoded
#         del residual
#         del ablated_logits
#         del signed_drop
#         del abs_drop

#         cleanup_memory()

#     all_signed_drops = torch.cat(all_signed_drops)
#     all_abs_drops = torch.cat(all_abs_drops)
#     all_clean_logits = torch.cat(all_clean_logits)
#     all_ablated_logits = torch.cat(all_ablated_logits)

#     return {
#         "mean_signed_drop": all_signed_drops.mean().item(),
#         "std_signed_drop": all_signed_drops.std(unbiased=False).item(),
#         "mean_abs_drop": all_abs_drops.mean().item(),
#         "std_abs_drop": all_abs_drops.std(unbiased=False).item(),
#         "mean_clean_logit": all_clean_logits.mean().item(),
#         "mean_ablated_logit": all_ablated_logits.mean().item(),
#         "n_examples": int(len(all_signed_drops)),
#     }


# def get_topk_features_for_layer_emotion(
#     df,
#     layer_idx,
#     emotion,
#     k,
#     use_abs_ranking=True,
# ):
#     """
#     Pick top-k features within one layer for one target emotion.
#     Ranking is based on existing single-feature ablation effect.
#     """
#     logit_col = f"logit_{emotion}"

#     df_layer = df[df["layer_idx"] == int(layer_idx)].copy()
#     df_layer[logit_col] = pd.to_numeric(df_layer[logit_col], errors="coerce")
#     df_layer = df_layer.dropna(subset=[logit_col])

#     if len(df_layer) == 0:
#         return [], df_layer

#     if use_abs_ranking:
#         df_layer["_rank_score"] = df_layer[logit_col].abs()
#     else:
#         df_layer["_rank_score"] = df_layer[logit_col]

#     df_layer = df_layer.sort_values("_rank_score", ascending=False)

#     selected = df_layer.head(int(k)).copy()

#     return selected["feature_idx"].astype(int).tolist(), selected


# def load_existing_results(out_path):
#     """
#     Resume from existing output if it exists.
#     """
#     if not os.path.exists(out_path):
#         return [], set()

#     existing = pd.read_csv(out_path)

#     if len(existing) == 0:
#         return [], set()

#     required_cols = ["layer_idx", "target_emotion", "top_k"]
#     missing = [c for c in required_cols if c not in existing.columns]
#     if missing:
#         raise ValueError(
#             f"Existing output is missing columns {missing}. "
#             f"Move or delete this file before resuming: {out_path}"
#         )

#     done_keys = set(
#         zip(
#             existing["layer_idx"].astype(int),
#             existing["target_emotion"].astype(str),
#             existing["top_k"].astype(int),
#         )
#     )

#     results = existing.to_dict("records")

#     print(f"Resuming from: {out_path}")
#     print(f"Already completed rows: {len(done_keys)}")

#     return results, done_keys


# # ============================================================
# # Main
# # ============================================================

# def main():
#     if not torch.cuda.is_available():
#         raise RuntimeError("CUDA is required for this script.")

#     # -----------------------
#     # Load feature file
#     # -----------------------
#     df = pd.read_csv(FEATURE_CSV_PATH)

#     # keep only emotion-specific features
#     df = df[df["entropy_group"] == "emotion_specific"].copy()

#     required_cols = ["layer_idx", "feature_idx"] + [f"logit_{e}" for e in EMOTIONS]
#     missing = [c for c in required_cols if c not in df.columns]
#     if missing:
#         raise ValueError(f"Missing columns in feature file: {missing}")

#     df["layer_idx"] = df["layer_idx"].astype(int)
#     df["feature_idx"] = df["feature_idx"].astype(int)

#     for e in EMOTIONS:
#         df[f"logit_{e}"] = pd.to_numeric(df[f"logit_{e}"], errors="coerce")

#     layers = sorted(df["layer_idx"].unique().tolist())

#     print("Feature df shape:", df.shape)
#     print("Layers:", layers)
#     print("Top-k list:", TOP_K_LIST)
#     print("Output path:", OUT_PATH)

#     # -----------------------
#     # Resume existing result
#     # -----------------------
#     results, done_keys = load_existing_results(OUT_PATH)

#     # -----------------------
#     # Build examples
#     # -----------------------
#     examples_by_emotion = build_examples_by_emotion(
#         dataset_csv_path=DATASET_CSV_PATH,
#         emotions=EMOTIONS,
#         text_col=TEXT_COL,
#         label_col=LABEL_COL,
#         sample_size_per_emotion=SAMPLE_SIZE_PER_EMOTION,
#         random_state=RANDOM_STATE,
#         apply_prompt_template=APPLY_PROMPT_TEMPLATE,
#     )

#     # -----------------------
#     # Load model and SAEs
#     # -----------------------
#     llm = load_llm(
#         model_name=MODEL_NAME,
#         device=DEVICE,
#         dtype=DTYPE,
#         cache_dir=CACHE_DIR,
#     )

#     emotion_to_token = build_emotion_token_map(
#         llm=llm,
#         emotions=EMOTIONS,
#     )

#     sae_cache = SAECache(
#         model_name=MODEL_NAME,
#         device=DEVICE,
#     )

#     # -----------------------
#     # Run top-k joint ablation
#     # -----------------------
#     for layer_idx in tqdm(layers, desc="Layers"):
#         sae = sae_cache.load(layer_idx)

#         for emotion in tqdm(EMOTIONS, desc=f"Layer {layer_idx} emotions", leave=False):
#             examples = examples_by_emotion[emotion]
#             target_token = emotion_to_token[emotion]

#             if len(examples) == 0:
#                 print(f"Skipping {emotion}: no examples.")
#                 continue

#             for k in TOP_K_LIST:
#                 key = (int(layer_idx), str(emotion), int(k))

#                 if key in done_keys:
#                     continue

#                 feature_indices, selected_df = get_topk_features_for_layer_emotion(
#                     df=df,
#                     layer_idx=layer_idx,
#                     emotion=emotion,
#                     k=k,
#                     use_abs_ranking=USE_ABS_RANKING,
#                 )

#                 if len(feature_indices) < k:
#                     print(
#                         f"Stopping L{layer_idx} {emotion}: only {len(feature_indices)} "
#                         f"features available, fewer than top{k}."
#                     )
#                     break

#                 stats = compute_topk_joint_logit_drop(
#                     llm=llm,
#                     sae=sae,
#                     examples=examples,
#                     layer_idx=layer_idx,
#                     feature_indices=feature_indices,
#                     target_token=target_token,
#                     batch_size=BATCH_SIZE,
#                 )

#                 logit_col = f"logit_{emotion}"

#                 row = {
#                     "layer_idx": int(layer_idx),
#                     "target_emotion": emotion,
#                     "top_k": int(k),
#                     "n_selected_features": int(len(feature_indices)),
#                     "features": ",".join(map(str, feature_indices)),
#                     "single_feature_abs_effect_sum": float(selected_df[logit_col].abs().sum()),
#                     "single_feature_abs_effect_mean": float(selected_df[logit_col].abs().mean()),
#                     "single_feature_abs_effect_min": float(selected_df[logit_col].abs().min()),
#                     "single_feature_abs_effect_max": float(selected_df[logit_col].abs().max()),
#                     **stats,
#                 }

#                 results.append(row)
#                 done_keys.add(key)

#                 result_df_partial = pd.DataFrame(results)

#                 if SAVE_EVERY_ROW:
#                     safe_save_csv(result_df_partial, OUT_PATH)

#                 print(
#                     f"L{layer_idx:02d} {emotion:8s} top{k:<3d} "
#                     f"signed_drop={stats['mean_signed_drop']:.6f} "
#                     f"abs_drop={stats['mean_abs_drop']:.6f} "
#                     f"features={feature_indices}"
#                 )

#         # Save after each layer too
#         result_df_partial = pd.DataFrame(results)
#         safe_save_csv(result_df_partial, OUT_PATH)
#         print(f"Saved after layer {layer_idx}: {OUT_PATH}")

#     # -----------------------
#     # Final save and summary
#     # -----------------------
#     result_df = pd.DataFrame(results)
#     safe_save_csv(result_df, OUT_PATH)

#     print("\nFinal result shape:", result_df.shape)
#     print("Saved to:", OUT_PATH)

#     print("\nMean signed drop by emotion and k:")
#     print(
#         result_df
#         .groupby(["target_emotion", "top_k"])["mean_signed_drop"]
#         .mean()
#         .unstack()
#     )

#     print("\nMean signed drop by layer and k:")
#     print(
#         result_df
#         .groupby(["layer_idx", "top_k"])["mean_signed_drop"]
#         .mean()
#         .unstack()
#     )

#     return result_df


# result_df = main()

In [ ]:
# import pandas as pd
# import numpy as np

# # ============================================================
# # Config
# # ============================================================

# FEATURE_CSV_PATH = (
#     "data/google-gemma-2-2b-feature-act/"
#     "core_emotion_features_filled_logits_signed_entropy.csv"
# )

# JOINT_ABLATION_CSV_PATH = (
#     "data/google-gemma-2-2b-feature-act/"
#     "topk_joint_ablation_logit_drop.csv"
# )

# OUT_PATH = (
#     "data/google-gemma-2-2b-feature-act/"
#     "auto_selected_emotion_specific_topk_features.csv"
# )

# EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

# MIN_SIGNED_DROP = 0.1
# COVERAGE_RATIO = 0.80


# # ============================================================
# # Load data
# # ============================================================

# feature_df = pd.read_csv(FEATURE_CSV_PATH)
# joint_df = pd.read_csv(JOINT_ABLATION_CSV_PATH)

# if "entropy_group" not in feature_df.columns:
#     raise ValueError("Missing column in feature file: entropy_group")

# feature_df = feature_df[
#     feature_df["entropy_group"] == "emotion_specific"
# ].copy()

# HAS_DESCRIPTION = "description" in feature_df.columns


# # ============================================================
# # Validate columns
# # ============================================================

# required_joint_cols = [
#     "layer_idx",
#     "target_emotion",
#     "top_k",
#     "mean_signed_drop",
#     "features",
# ]

# missing = [c for c in required_joint_cols if c not in joint_df.columns]
# if missing:
#     raise ValueError(f"Missing columns in joint ablation file: {missing}")

# required_feature_cols = [
#     "layer_idx",
#     "feature_idx",
#     "entropy_group",
# ] + [f"logit_{e}" for e in EMOTIONS]

# missing = [c for c in required_feature_cols if c not in feature_df.columns]
# if missing:
#     raise ValueError(f"Missing columns in feature file: {missing}")


# # ============================================================
# # Clean types
# # ============================================================

# feature_df["layer_idx"] = feature_df["layer_idx"].astype(int)
# feature_df["feature_idx"] = feature_df["feature_idx"].astype(int)

# joint_df["layer_idx"] = joint_df["layer_idx"].astype(int)
# joint_df["top_k"] = joint_df["top_k"].astype(int)
# joint_df["target_emotion"] = joint_df["target_emotion"].astype(str)
# joint_df["mean_signed_drop"] = pd.to_numeric(
#     joint_df["mean_signed_drop"],
#     errors="coerce",
# )

# for e in EMOTIONS:
#     feature_df[f"logit_{e}"] = pd.to_numeric(
#         feature_df[f"logit_{e}"],
#         errors="coerce",
#     )


# # ============================================================
# # Helpers
# # ============================================================

# def parse_feature_list(x):
#     if pd.isna(x):
#         return []

#     x = str(x).strip()

#     if x == "":
#         return []

#     return [int(v.strip()) for v in x.split(",") if v.strip() != ""]


# def choose_min_k_for_layer_emotion(df_group):
#     df_valid = df_group.dropna(subset=["mean_signed_drop"]).copy()
#     df_valid = df_valid[df_valid["mean_signed_drop"] >= MIN_SIGNED_DROP].copy()

#     if len(df_valid) == 0:
#         return None

#     max_drop = df_valid["mean_signed_drop"].max()
#     threshold = COVERAGE_RATIO * max_drop

#     df_valid = df_valid.sort_values("top_k", ascending=True)
#     df_hit = df_valid[df_valid["mean_signed_drop"] >= threshold].copy()

#     if len(df_hit) == 0:
#         return None

#     selected = df_hit.iloc[0].copy()
#     selected["max_signed_drop"] = max_drop
#     selected["coverage_threshold"] = threshold
#     selected["coverage_ratio_actual"] = selected["mean_signed_drop"] / max_drop

#     return selected


# # ============================================================
# # Select rows
# # ============================================================

# rows = []

# for (layer_idx, emotion), df_group in joint_df.groupby(
#     ["layer_idx", "target_emotion"]
# ):
#     if emotion not in EMOTIONS:
#         continue

#     selected = choose_min_k_for_layer_emotion(df_group)

#     if selected is None:
#         rows.append(
#             {
#                 "layer_idx": int(layer_idx),
#                 "target_emotion": emotion,
#                 "selected_top_k": 0,
#                 "selected_mean_signed_drop": np.nan,
#                 "max_signed_drop": np.nan,
#                 "coverage_threshold": np.nan,
#                 "coverage_ratio_actual": np.nan,
#                 "feature_idx": np.nan,
#                 "selection_status": "no_k_passed_threshold",
#             }
#         )
#         continue

#     selected_features = parse_feature_list(selected["features"])

#     for feature_idx in selected_features:
#         matched = feature_df[
#             (feature_df["layer_idx"] == int(layer_idx))
#             & (feature_df["feature_idx"] == int(feature_idx))
#         ]

#         if len(matched) == 0:
#             continue

#         feature_row = matched.iloc[0]

#         out = {
#             "layer_idx": int(layer_idx),
#             "target_emotion": emotion,
#             "selected_top_k": int(selected["top_k"]),
#             "selected_mean_signed_drop": float(selected["mean_signed_drop"]),
#             "max_signed_drop": float(selected["max_signed_drop"]),
#             "coverage_threshold": float(selected["coverage_threshold"]),
#             "coverage_ratio_actual": float(selected["coverage_ratio_actual"]),
#             "feature_idx": int(feature_idx),
#             "entropy_group": feature_row["entropy_group"],
#             "selection_status": "selected",
#         }

#         for e in EMOTIONS:
#             out[f"logit_{e}"] = feature_row[f"logit_{e}"]

#         if HAS_DESCRIPTION:
#             out["description"] = feature_row["description"]

#         rows.append(out)


# # ============================================================
# # Add shared-feature info within selected output
# # ============================================================

# result_df = pd.DataFrame(rows)

# if len(result_df) > 0:
#     selected_only = result_df[result_df["selection_status"] == "selected"].copy()

#     shared_map = (
#         selected_only
#         .groupby(["layer_idx", "feature_idx"])["target_emotion"]
#         .apply(lambda x: ",".join(sorted(set(x), key=EMOTIONS.index)))
#         .reset_index()
#         .rename(columns={"target_emotion": "selected_emotions_for_feature"})
#     )

#     count_map = (
#         selected_only
#         .groupby(["layer_idx", "feature_idx"])["target_emotion"]
#         .nunique()
#         .reset_index()
#         .rename(columns={"target_emotion": "num_selected_emotions_for_feature"})
#     )

#     result_df = result_df.merge(
#         shared_map,
#         on=["layer_idx", "feature_idx"],
#         how="left",
#     )

#     result_df = result_df.merge(
#         count_map,
#         on=["layer_idx", "feature_idx"],
#         how="left",
#     )

#     result_df["selected_emotions_for_feature"] = result_df[
#         "selected_emotions_for_feature"
#     ].fillna("")

#     result_df["num_selected_emotions_for_feature"] = result_df[
#         "num_selected_emotions_for_feature"
#     ].fillna(0).astype(int)

#     result_df = result_df.sort_values(
#         by=[
#             "layer_idx",
#             "target_emotion",
#             "selected_top_k",
#             "num_selected_emotions_for_feature",
#             "feature_idx",
#         ],
#         ascending=[True, True, True, False, True],
#     ).reset_index(drop=True)

# result_df.to_csv(OUT_PATH, index=False)

# print("Final result shape:", result_df.shape)
# print("Saved to:", OUT_PATH)
# print(result_df)

In [ ]:
# SECOND_OUT_PATH = (
#     "data/google-gemma-2-2b-feature-act/"
#     "auto_selected_unique_features_with_affected_emotions.csv"
# )

# selected_only = result_df[result_df["selection_status"] == "selected"].copy()

# affected_map = (
#     selected_only
#     .groupby(["layer_idx", "feature_idx"])["target_emotion"]
#     .apply(lambda x: ",".join(sorted(set(x), key=EMOTIONS.index)))
#     .reset_index()
#     .rename(columns={"target_emotion": "affected_emotions"})
# )

# feature_rows = (
#     selected_only
#     .drop_duplicates(subset=["layer_idx", "feature_idx"])
#     .copy()
# )

# keep_cols = [
#     "layer_idx",
#     "feature_idx",
# ] + [f"logit_{e}" for e in EMOTIONS]

# if "description" in feature_rows.columns:
#     keep_cols.append("description")

# feature_rows = feature_rows[keep_cols]

# feature_rows = feature_rows.merge(
#     affected_map,
#     on=["layer_idx", "feature_idx"],
#     how="left",
# )

# feature_rows = feature_rows.sort_values(
#     by=["layer_idx", "feature_idx"]
# ).reset_index(drop=True)

# feature_rows.to_csv(SECOND_OUT_PATH, index=False)

# print("Saved second output to:", SECOND_OUT_PATH)
# print("Second output shape:", feature_rows.shape)
# print(feature_rows)

In [ ]:
import os

CACHE_ROOT = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_ROOT}/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_ROOT}/transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{CACHE_ROOT}/hub"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT

In [ ]:
import torch

from sae_emotion.logit_ablation_utils import (
    model_safe_name,
    run_topk_joint_ablation_pipeline,
    run_topk_selection_pipeline,
)

MODEL_NAME = "google/gemma-2-2b"
SAFE_NAME = model_safe_name(MODEL_NAME)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

FEATURE_CSV_PATH = (
    "data/google-gemma-2-2b-feature-act/"
    "core_emotion_features_filled_logits_signed_entropy.csv"
)

DATASET_CSV_PATH = f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"

JOINT_OUT_PATH = (
    "data/google-gemma-2-2b-feature-act/"
    "topk_joint_ablation_logit_drop.csv"
)

SELECTED_OUT_PATH = (
    "data/google-gemma-2-2b-feature-act/"
    "auto_selected_emotion_specific_topk_features.csv"
)

UNIQUE_OUT_PATH = (
    "data/google-gemma-2-2b-feature-act/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

# 1. Run top-k joint ablation
joint_df = run_topk_joint_ablation_pipeline(
    model_name=MODEL_NAME,
    feature_csv_path=FEATURE_CSV_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_csv_path=JOINT_OUT_PATH,
    emotions=EMOTIONS,
    top_k_list=[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20],
    cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub")),
    sample_size_per_emotion=300,
    batch_size=128,
    apply_prompt_template=False,
    use_abs_ranking=False,
    device="cuda",
    dtype=torch.bfloat16,
)

# 2. Select minimal top-k core features
selected_df, unique_df = run_topk_selection_pipeline(
    feature_csv_path=FEATURE_CSV_PATH,
    joint_ablation_csv_path=JOINT_OUT_PATH,
    selected_output_csv_path=SELECTED_OUT_PATH,
    unique_output_csv_path=UNIQUE_OUT_PATH,
    emotions=EMOTIONS,
    min_signed_drop=0.1,
    coverage_ratio=0.80,
)

In [ ]:
# import os
# import textwrap
# import numpy as np
# import pandas as pd
# import plotly.graph_objects as go

# # =========================
# # Config
# # =========================
# FEATURE_PATH = (
#     "data/google-gemma-2-2b-feature-act/"
#     "core_emotion_features_filled_logits_signed_entropy_top0.25.csv"
# )

# OUTPUT_HTML = (
#     "data/google-gemma-2-2b-feature-act/"
#     "visuals/emotion_feature_nodes_only_top0.25.html"
# )

# EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

# EMOTION_COLORS = {
#     "joy": "#F58518",
#     "surprise": "#72B7B2",
#     "disgust": "#54A24B",
#     "anger": "#E45756",
#     "sadness": "#4C78A8",
#     "fear": "#B279A2",
# }

# NODE_MIN_SIZE = 16
# NODE_MAX_SIZE = 44
# EMOTION_BAND_GAP = 14.0
# RANDOM_SEED = 42

# # =========================
# # Load
# # =========================
# features = pd.read_csv(FEATURE_PATH)

# features["layer_idx"] = features["layer_idx"].astype(int)
# features["feature_idx"] = features["feature_idx"].astype(int)

# features["node_id"] = (
#     "L"
#     + features["layer_idx"].astype(str)
#     + "-F"
#     + features["feature_idx"].astype(str)
# )

# # =========================
# # Shared detection
# # =========================
# features["is_shared"] = (
#     features["entropy_group"] == "shared_broad"
# )

# # =========================
# # Logit helpers
# # =========================
# LOGIT_COLS = [f"logit_{e}" for e in EMOTIONS]

# features["max_abs_logit"] = (
#     features[LOGIT_COLS].abs().max(axis=1)
# )

# def get_top_two_logit_emotions(row):
#     vals = []

#     for emo in EMOTIONS:
#         vals.append((emo, abs(row[f"logit_{emo}"])))

#     vals = sorted(vals, key=lambda x: x[1], reverse=True)

#     return vals[0][0], vals[1][0]

# features[["top_logit_emotion", "second_logit_emotion"]] = features.apply(
#     lambda row: pd.Series(get_top_two_logit_emotions(row)),
#     axis=1
# )

# features["dominant_emotion"] = features["top_logit_emotion"]

# # =========================
# # Node layout
# # =========================
# emotion_y_base = {
#     emo: i * EMOTION_BAND_GAP
#     for i, emo in enumerate(EMOTIONS)
# }

# def get_y_base(row):
#     if row["is_shared"]:
#         return np.mean(list(emotion_y_base.values()))

#     return emotion_y_base[row["dominant_emotion"]]

# features["layout_group"] = np.where(
#     features["is_shared"],
#     "shared",
#     features["dominant_emotion"]
# )

# features["y_base"] = features.apply(get_y_base, axis=1)

# features = features.sort_values(
#     ["layout_group", "layer_idx", "feature_idx"]
# ).reset_index(drop=True)

# features["within_group_idx"] = features.groupby(
#     ["layout_group", "layer_idx"]
# ).cumcount()

# features["group_count"] = features.groupby(
#     ["layout_group", "layer_idx"]
# )["node_id"].transform("count")

# features["y_pos"] = (
#     features["y_base"]
#     + features["within_group_idx"]
#     - (features["group_count"] - 1) / 2
# )

# np.random.seed(RANDOM_SEED)

# features["x_pos"] = (
#     features["layer_idx"]
#     + np.random.uniform(-0.08, 0.08, len(features))
# )

# features["y_pos"] = (
#     features["y_pos"]
#     + np.random.uniform(-0.08, 0.08, len(features))
# )

# # =========================
# # Node sizes
# # =========================
# v = features["max_abs_logit"].to_numpy()
# v = np.clip(v, 1e-6, None)

# v_log = np.log(v)

# v_norm = (
#     (v_log - v_log.min())
#     / (v_log.max() - v_log.min() + 1e-8)
# )

# features["node_size"] = (
#     NODE_MIN_SIZE
#     + v_norm * (NODE_MAX_SIZE - NODE_MIN_SIZE)
# )

# # =========================
# # Hover text
# # =========================
# def format_logit_weights(row):
#     return "<br>".join([
#         f"{emo}: {row[f'logit_{emo}']:.4f}"
#         for emo in EMOTIONS
#     ])

# hover_texts = []

# for _, row in features.iterrows():

#     desc = str(row["description"]) if "description" in row else ""

#     desc = "<br>".join(
#         textwrap.wrap(desc, width=80)
#     )

#     hover_texts.append(
#         f"<b>{row['node_id']}</b><br>"
#         f"Layer: {row['layer_idx']}<br>"
#         f"Feature: {row['feature_idx']}<br>"
#         f"Entropy group: {row['entropy_group']}<br>"
#         f"Normalized entropy: {row['normalized_entropy']:.3f}<br>"
#         f"Max abs logit: {row['max_abs_logit']:.4f}<br>"
#         f"<br><b>Logit weights:</b><br>"
#         f"{format_logit_weights(row)}<br>"
#         f"<br><b>Description:</b><br>"
#         f"{desc}"
#     )

# features["hover"] = hover_texts

# # =========================
# # Non-shared nodes
# # =========================
# node_traces = []

# for emotion in EMOTIONS:

#     sub = features[
#         (~features["is_shared"])
#         & (features["dominant_emotion"] == emotion)
#     ]

#     if len(sub) == 0:
#         continue

#     node_traces.append(
#         go.Scatter(
#             x=sub["x_pos"],
#             y=sub["y_pos"],
#             mode="markers+text",
#             text=sub["node_id"],
#             textposition="middle center",
#             textfont=dict(
#                 size=7,
#                 color="white"
#             ),
#             hoverinfo="text",
#             hovertext=sub["hover"],
#             marker=dict(
#                 size=sub["node_size"],
#                 color=EMOTION_COLORS[emotion],
#                 line=dict(
#                     width=1,
#                     color="black"
#                 ),
#             ),
#             name=emotion,
#         )
#     )

# # =========================
# # Shared nodes
# # =========================
# shared_nodes = features[features["is_shared"]]

# shared_trace = go.Scatter(
#     x=shared_nodes["x_pos"],
#     y=shared_nodes["y_pos"],
#     mode="markers+text",
#     text=shared_nodes["node_id"],
#     textposition="middle center",
#     textfont=dict(
#         size=7,
#         color="white"
#     ),
#     hoverinfo="text",
#     hovertext=shared_nodes["hover"],
#     marker=dict(
#         size=shared_nodes["node_size"],
#         color=shared_nodes["top_logit_emotion"].map(EMOTION_COLORS),
#         line=dict(
#             width=3,
#             color=shared_nodes["second_logit_emotion"].map(EMOTION_COLORS),
#         ),
#     ),
#     name="shared_broad",
# )

# # =========================
# # Emotion labels
# # =========================
# annotations = []

# for emotion in EMOTIONS:

#     annotations.append(
#         dict(
#             x=1.02,
#             y=emotion_y_base[emotion],
#             xref="paper",
#             yref="y",
#             text=f"<b>{emotion}</b>",
#             showarrow=False,
#             font=dict(
#                 size=14,
#                 color=EMOTION_COLORS[emotion]
#             ),
#             xanchor="left",
#         )
#     )

# annotations.append(
#     dict(
#         x=1.02,
#         y=np.mean(list(emotion_y_base.values())),
#         xref="paper",
#         yref="y",
#         text="<b>shared / broad<br>features</b>",
#         showarrow=False,
#         font=dict(size=11),
#         xanchor="left",
#     )
# )

# # =========================
# # Plot
# # =========================
# fig = go.Figure(
#     data=node_traces + [shared_trace]
# )

# fig.update_layout(
#     title=(
#         "Emotion Feature Nodes<br>"
#         "<sup>Shared nodes: fill=largest logit emotion, border=second largest</sup>"
#     ),
#     xaxis=dict(
#         title="Layer"
#     ),
#     yaxis=dict(
#         showticklabels=False
#     ),
#     width=1600,
#     height=1100,
#     annotations=annotations,
# )

# fig.show()

# os.makedirs(
#     os.path.dirname(OUTPUT_HTML),
#     exist_ok=True
# )

# fig.write_html(OUTPUT_HTML)

# print("Saved to:", OUTPUT_HTML)

## Build Feature Graph

In [ ]:
# import os
# import torch
# import pandas as pd
# from collections import defaultdict
# from tqdm import tqdm


# # ============================================================
# # Config
# # ============================================================

# EMOTION_SPEC_PATH = "data/google-gemma-2-2b-feature-act/auto_selected_unique_features_with_affected_emotions.csv"
# OUTPUT_DIR = "data/google-gemma-2-2b-feature-act/emotion_feature_edges"

# EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

# MIN_SOURCE_TARGET_LAYER_GAP = 1
# USE_ABS_EFFECT = False          # False keeps signed direction
# CONDITION_ON_TARGET_ACTIVE = False
# TARGET_ACTIVE_THRESHOLD = 0.0   # use > 0.0 for SAE activation
# MIN_TARGET_ACTIVE_COUNT = 10

# os.makedirs(OUTPUT_DIR, exist_ok=True)


# # ============================================================
# # Zero-ablation helper
# # ============================================================

# def zero_ablate(sae, source_sae_features, feature_idx, residual):
#     """
#     Reconstruct hidden state with one SAE feature zeroed out.
#     """
#     patched_features = source_sae_features.clone()
#     patched_features[:, feature_idx] = 0.0

#     patched_hidden = sae.decode(patched_features) + residual
#     return patched_hidden


# # ============================================================
# # Edge patching between one source layer and one target layer
# # ============================================================

# def edge_patching(
#     llm,
#     source_sae,
#     target_sae,
#     examples,
#     source_layer: int,
#     source_features,
#     target_layer: int,
#     target_features,
#     use_abs_effect: bool = False,
#     condition_on_target_active: bool = False,
#     target_active_threshold: float = 0.0,
#     min_target_active_count: int = 5,
# ):
#     """
#     Estimate directed feature edges from source_features to target_features.

#     Edge weight:
#         clean target activation - target activation after source feature ablation

#     If use_abs_effect=True:
#         abs(clean - ablated)

#     If condition_on_target_active=True:
#         average only over examples where the clean target feature is active.
#     """

#     source_features = list(map(int, source_features))
#     target_features = list(map(int, target_features))

#     edges = []

#     # ------------------------------------------------------------
#     # 1. Clean activations
#     # ------------------------------------------------------------
#     with torch.no_grad():
#         with llm.trace(examples):
#             source_hidden = llm.model.layers[source_layer].output[:, -1, :].save()
#             target_hidden = llm.model.layers[target_layer].output[:, -1, :].save()

#     source_sae_features = source_sae.encode(source_hidden)
#     source_hidden_decoded = source_sae.decode(source_sae_features)
#     source_sae_residual = source_hidden - source_hidden_decoded

#     target_sae_features_clean = target_sae.encode(target_hidden)
#     clean_targets = target_sae_features_clean[:, target_features]  # [B, T]

#     # ------------------------------------------------------------
#     # 2. Ablate each source feature and collect target hidden states
#     # ------------------------------------------------------------
#     target_hiddens = []

#     for source_feat in source_features:
#         with torch.no_grad():
#             with llm.trace(examples):
#                 llm.model.layers[source_layer].output[:, -1, :] = zero_ablate(
#                     sae=source_sae,
#                     source_sae_features=source_sae_features,
#                     feature_idx=source_feat,
#                     residual=source_sae_residual,
#                 )
#                 tgt_hidden = llm.model.layers[target_layer].output[:, -1, :].save()
#                 target_hiddens.append(tgt_hidden)

#     target_hiddens = torch.stack(target_hiddens)  # [S, B, d]

#     # ------------------------------------------------------------
#     # 3. Encode ablated target hidden states
#     # ------------------------------------------------------------
#     num_source_feats = len(source_features)
#     batch_size = target_hiddens.shape[1]

#     target_hiddens_flat = target_hiddens.reshape(num_source_feats * batch_size, -1)

#     with torch.no_grad():
#         target_sae_features_flat = target_sae.encode(target_hiddens_flat)

#     target_sae_features = target_sae_features_flat.reshape(num_source_feats, batch_size, -1)
#     ablated_targets = target_sae_features[:, :, target_features]  # [S, B, T]

#     clean_targets_expanded = clean_targets.unsqueeze(0)  # [1, B, T]

#     diffs = clean_targets_expanded - ablated_targets

#     if use_abs_effect:
#         diffs = diffs.abs()

#     # ------------------------------------------------------------
#     # 4. Aggregate into edges
#     # ------------------------------------------------------------
#     for s_i, source_feat in enumerate(source_features):
#         for t_i, target_feat in enumerate(target_features):

#             target_clean = clean_targets[:, t_i]
#             edge_diffs = diffs[s_i, :, t_i]

#             if condition_on_target_active:
#                 mask = target_clean > target_active_threshold
#                 n_used = int(mask.sum().item())

#                 if n_used < min_target_active_count:
#                     mean_effect = float("nan")
#                     mean_abs_effect = float("nan")
#                 else:
#                     selected_diffs = edge_diffs[mask]
#                     mean_effect = float(selected_diffs.mean().item())
#                     mean_abs_effect = float(selected_diffs.abs().mean().item())
#             else:
#                 n_used = int(batch_size)
#                 mean_effect = float(edge_diffs.mean().item())
#                 mean_abs_effect = float(edge_diffs.abs().mean().item())

#             clean_mean = float(target_clean.mean().item())
#             clean_active_count = int((target_clean > target_active_threshold).sum().item())

#             if abs(clean_mean) > 1e-8:
#                 normalized_effect = mean_effect / (abs(clean_mean) + 1e-8)
#             else:
#                 normalized_effect = float("nan")

#             edges.append(
#                 {
#                     "source_layer": source_layer,
#                     "source_feature": source_feat,
#                     "target_layer": target_layer,
#                     "target_feature": target_feat,
#                     "n_examples_used": n_used,
#                     "clean_target_mean": clean_mean,
#                     "clean_target_active_count": clean_active_count,
#                     "mean_effect": mean_effect,
#                     "mean_abs_effect": mean_abs_effect,
#                     "normalized_effect": normalized_effect,
#                 }
#             )

#     return edges


# # ============================================================
# # Build emotion-specific graphs
# # ============================================================

# def parse_affected_emotions(x):
#     if pd.isna(x):
#         return []

#     x = str(x).strip()

#     if x == "":
#         return []

#     return [
#         e.strip()
#         for e in x.split(",")
#         if e.strip() in EMOTIONS
#     ]


# def build_emotion_feature_graphs(
#     llm,
#     emotion_spec,
#     examples_by_emotion,
#     output_dir=OUTPUT_DIR,
# ):
#     required_cols = ["layer_idx", "feature_idx", "affected_emotions"]
#     missing = [c for c in required_cols if c not in emotion_spec.columns]
#     if missing:
#         raise ValueError(f"emotion_spec is missing columns: {missing}")

#     emotion_spec = emotion_spec.copy()

#     emotion_spec["layer_idx"] = emotion_spec["layer_idx"].astype(int)
#     emotion_spec["feature_idx"] = emotion_spec["feature_idx"].astype(int)

#     emotion_spec["affected_emotion_list"] = emotion_spec["affected_emotions"].apply(
#         parse_affected_emotions
#     )

#     emotion_spec["num_affected_emotions"] = emotion_spec["affected_emotion_list"].apply(len)

#     all_edges = []

#     for emotion in EMOTIONS:
#         print(f"\nBuilding graph for emotion: {emotion}")

#         if emotion not in examples_by_emotion:
#             print(f"Skipping {emotion}: no examples found.")
#             continue

#         examples = examples_by_emotion[emotion]

#         emotion_features = emotion_spec[
#             emotion_spec["affected_emotion_list"].apply(lambda xs: emotion in xs)
#         ].copy()

#         if len(emotion_features) == 0:
#             print(f"Skipping {emotion}: no features found.")
#             continue

#         layer_to_features = (
#             emotion_features
#             .groupby("layer_idx")["feature_idx"]
#             .apply(lambda x: sorted(list(map(int, x))))
#             .to_dict()
#         )

#         layers = sorted(layer_to_features.keys())

#         emotion_edges = []

#         for source_layer in tqdm(layers, desc=f"{emotion} source layers"):
#             source_sae = load_sae(source_layer)
#             source_features = layer_to_features[source_layer]

#             for target_layer in layers:
#                 if target_layer - source_layer < MIN_SOURCE_TARGET_LAYER_GAP:
#                     continue

#                 target_sae = load_sae(target_layer)
#                 target_features = layer_to_features[target_layer]

#                 edges = edge_patching(
#                     llm=llm,
#                     source_sae=source_sae,
#                     target_sae=target_sae,
#                     examples=examples,
#                     source_layer=int(source_layer),
#                     source_features=source_features,
#                     target_layer=int(target_layer),
#                     target_features=target_features,
#                     use_abs_effect=USE_ABS_EFFECT,
#                     condition_on_target_active=CONDITION_ON_TARGET_ACTIVE,
#                     target_active_threshold=TARGET_ACTIVE_THRESHOLD,
#                     min_target_active_count=MIN_TARGET_ACTIVE_COUNT,
#                 )

#                 for edge in edges:
#                     edge["conditioning_emotion"] = emotion

#                     src_row = emotion_features[
#                         (emotion_features["layer_idx"] == edge["source_layer"])
#                         & (emotion_features["feature_idx"] == edge["source_feature"])
#                     ].iloc[0]

#                     tgt_row = emotion_features[
#                         (emotion_features["layer_idx"] == edge["target_layer"])
#                         & (emotion_features["feature_idx"] == edge["target_feature"])
#                     ].iloc[0]

#                     edge["source_affected_emotions"] = src_row["affected_emotions"]
#                     edge["source_num_affected_emotions"] = int(src_row["num_affected_emotions"])

#                     edge["target_affected_emotions"] = tgt_row["affected_emotions"]
#                     edge["target_num_affected_emotions"] = int(tgt_row["num_affected_emotions"])

#                 emotion_edges.extend(edges)

#                 clear_cuda()

#         emotion_edges_df = pd.DataFrame(emotion_edges)

#         emotion_output_path = os.path.join(
#             output_dir,
#             f"{emotion}_affected_feature_edges.csv"
#         )

#         emotion_edges_df.to_csv(emotion_output_path, index=False)
#         print(f"Saved {emotion} edges to: {emotion_output_path}")

#         all_edges.append(emotion_edges_df)

#     all_edges_df = (
#         pd.concat(all_edges, ignore_index=True)
#         if all_edges
#         else pd.DataFrame()
#     )

#     all_output_path = os.path.join(
#         output_dir,
#         "all_affected_feature_edges.csv"
#     )

#     all_edges_df.to_csv(all_output_path, index=False)

#     print(f"\nSaved all edges to: {all_output_path}")
#     return all_edges_df

In [ ]:
# # --------------------
# # SAE helpers
# # --------------------

# _sae_cache = {}

# def load_sae(layer_idx):
#     layer_idx = int(layer_idx)

#     if layer_idx in _sae_cache:
#         return _sae_cache[layer_idx]

#     if layer_idx < 0 or layer_idx >= 26:
#         raise ValueError(f"Invalid layer index: {layer_idx}. Model has 26 layers.")

#     release = "gemma-scope-2b-pt-res-canonical"
#     sae_id = f"layer_{layer_idx}/width_16k/canonical"

#     sae = SAE.from_pretrained(
#         release,
#         sae_id,
#         device=device
#     )

#     _sae_cache[layer_idx] = sae
#     print(f"Loaded SAE: {sae_id}")

#     return sae


# def clear_cuda():
#     gc.collect()
#     torch.cuda.empty_cache()


# def zero_ablate(sae, source_sae_features, feature_idx, residual):
#     modified_sae_features = source_sae_features.clone()
#     modified_sae_features[:, int(feature_idx)] = 0.0
#     modified_activations = sae.decode(modified_sae_features)
#     return modified_activations + residual

In [ ]:
# import random

# # --------------------
# # Load dataset
# # --------------------
# dataset = pd.read_csv("data/combined_zeroshot_filtered_google-gemma-2-2b.csv")

# # keep only needed columns
# dataset = dataset[["input_text", "emotion"]].dropna()

# # normalize labels
# dataset["emotion"] = dataset["emotion"].str.lower().str.strip()

# # target emotions
# emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

# # --------------------
# # Build examples per emotion
# # --------------------
# examples_by_emotion = {
#     emotion: dataset.loc[dataset["emotion"] == emotion, "input_text"].tolist()
#     for emotion in emotions
# }

# # check original counts
# print("Original counts:")
# for emotion, examples in examples_by_emotion.items():
#     print(emotion, len(examples))

# # --------------------
# # Balanced sampling (300 per emotion)
# # --------------------
# sample_size = 300
# random.seed(42)

# examples_by_emotion_balanced = {}

# for emotion in emotions:
#     examples = examples_by_emotion[emotion]

#     if len(examples) >= sample_size:
#         sampled = random.sample(examples, sample_size)
#     else:
#         # if not enough, use all (no oversampling)
#         sampled = examples

#     examples_by_emotion_balanced[emotion] = sampled

# # replace original
# examples_by_emotion = examples_by_emotion_balanced

# # check balanced counts
# print("\nBalanced counts:")
# for emotion, examples in examples_by_emotion.items():
#     print(emotion, len(examples))

In [ ]:
# import gc
# import torch
# import pandas as pd
# from tqdm import tqdm
# from nnsight import LanguageModel
# from sae_lens import SAE

# model_name = "google/gemma-2-2b"
# # --------------------
# # Load model
# # --------------------
# device = "cuda"
# llm = LanguageModel(
#     model_name,
#     device_map=device,
#     dtype=torch.bfloat16,
#     # cache_dir=cache_dir,
#     dispatch=True
# )

In [ ]:
# emotion_spec = pd.read_csv(EMOTION_SPEC_PATH)

# all_edges_df = build_emotion_feature_graphs(
#     llm=llm,
#     emotion_spec=emotion_spec,
#     examples_by_emotion=examples_by_emotion,
# )

In [ ]:
import torch
import pandas as pd

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_examples_by_emotion,
)

from sae_emotion.emotion_graph_utils import (
    run_emotion_feature_graph_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-2-2b"

EMOTION_SPEC_PATH = (
    "data/google-gemma-2-2b-feature-act/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

DATASET_CSV_PATH = (
    "data/combined_zeroshot_filtered_google-gemma-2-2b.csv"
)

OUTPUT_DIR = (
    "data/google-gemma-2-2b-feature-act/"
    "emotion_feature_edges"
)

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

DEVICE = "cuda"
DTYPE = torch.bfloat16

CACHE_DIR = os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub"))

# ============================================================
# Run graph building
# ============================================================

all_edges_df = run_emotion_feature_graph_pipeline(
    model_name=MODEL_NAME,

    emotion_spec_csv_path=EMOTION_SPEC_PATH,

    dataset_csv_path=DATASET_CSV_PATH,

    output_dir=OUTPUT_DIR,

    emotions=EMOTIONS,

    # dataset
    text_col="input_text",
    label_col="emotion",
    sample_size_per_emotion=300,
    random_state=42,
    apply_prompt_template=False,

    # model
    device=DEVICE,
    dtype=DTYPE,
    cache_dir=CACHE_DIR,

    # graph params
    min_source_target_layer_gap=1,

    # edge effect
    use_abs_effect=False,

    # optional conditioning
    condition_on_target_active=False,
    target_active_threshold=0.0,
    min_target_active_count=10,
)

print(all_edges_df.shape)
print(all_edges_df.head())

## Visualization

In [ ]:
import os
import textwrap
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================
# Paths / config
# =========================
FEATURE_PATH = "data/google-gemma-2-2b-feature-act/auto_selected_unique_features_with_affected_emotions.csv"

ALL_EDGE_PATH = (
    "data/google-gemma-2-2b-feature-act/"
    "emotion_feature_edges/all_affected_feature_edges.csv"
)

OUTPUT_HTML = (
    "data/google-gemma-2-2b-feature-act/"
    "visuals/core_features.html"
)

EMOTIONS = [
    "joy",
    "surprise",
    "disgust",
    "anger",
    "sadness",
    "fear",
]

EMOTION_COLORS = {
    "joy": "#F58518",
    "surprise": "#72B7B2",
    "disgust": "#54A24B",
    "anger": "#E45756",
    "sadness": "#4C78A8",
    "fear": "#B279A2",
}

NODE_MIN_SIZE = 18
NODE_MAX_SIZE = 48

EDGE_OPACITY = 0.18

RANDOM_SEED = 42
EMOTION_BAND_GAP = 14.0

# =========================
# Load data
# =========================
features = pd.read_csv(FEATURE_PATH)
edges = pd.read_csv(ALL_EDGE_PATH)

# =========================
# Parse affected emotions
# =========================
def parse_affected_emotions(x):
    if pd.isna(x):
        return []

    x = str(x).strip()

    x = (
        x.replace("[", "")
         .replace("]", "")
         .replace("'", "")
         .replace('"', "")
    )

    if x == "":
        return []

    return [
        e.strip()
        for e in x.split(",")
        if e.strip() in EMOTIONS
    ]


features = features[
    features["affected_emotions"].notna()
].copy()

features["feature_idx"] = features["feature_idx"].astype(int)
features["layer_idx"] = features["layer_idx"].astype(int)

features["node_id"] = (
    "L" + features["layer_idx"].astype(str)
    + "-F" + features["feature_idx"].astype(str)
)

features["affected_emotion_list"] = (
    features["affected_emotions"]
    .apply(parse_affected_emotions)
)

features["affected_emotion_list"] = (
    features["affected_emotion_list"]
    .apply(lambda xs: [e for e in EMOTIONS if e in xs])
)

features["num_affected_emotions"] = (
    features["affected_emotion_list"]
    .apply(len)
)

features["is_shared"] = (
    features["num_affected_emotions"] >= 2
)

features["affected_emotions_clean"] = (
    features["affected_emotion_list"]
    .apply(lambda x: ", ".join(x))
)

# =========================
# Edge node IDs
# =========================
edges["source_feature"] = edges["source_feature"].astype(int)
edges["target_feature"] = edges["target_feature"].astype(int)

edges["source_layer"] = edges["source_layer"].astype(int)
edges["target_layer"] = edges["target_layer"].astype(int)

edges["source_node_id"] = (
    "L" + edges["source_layer"].astype(str)
    + "-F" + edges["source_feature"].astype(str)
)

edges["target_node_id"] = (
    "L" + edges["target_layer"].astype(str)
    + "-F" + edges["target_feature"].astype(str)
)

valid_nodes = set(features["node_id"])

edges = edges[
    edges["source_node_id"].isin(valid_nodes)
    & edges["target_node_id"].isin(valid_nodes)
].copy()

# =========================
# Logit columns
# =========================
LOGIT_COLS = [f"logit_{e}" for e in EMOTIONS]

existing_logit_cols = [
    c for c in LOGIT_COLS
    if c in features.columns
]

if len(existing_logit_cols) == 0:
    raise ValueError("No logit_* columns found.")

if "max_abs_logit" not in features.columns:
    features["max_abs_logit"] = (
        features[existing_logit_cols]
        .abs()
        .max(axis=1)
    )

# =========================
# Top / second emotion
# =========================
def get_top_two_logit_emotions(row):
    vals = []

    for emo in EMOTIONS:
        col = f"logit_{emo}"

        if col in row.index:
            vals.append((emo, abs(row[col])))

    vals = sorted(vals, key=lambda x: x[1], reverse=True)

    top_emo = vals[0][0]

    if len(vals) > 1:
        second_emo = vals[1][0]
    else:
        second_emo = top_emo

    return top_emo, second_emo


features[["top_logit_emotion", "second_logit_emotion"]] = (
    features.apply(
        lambda row: pd.Series(
            get_top_two_logit_emotions(row)
        ),
        axis=1,
    )
)

# =========================
# Format logit weights
# =========================
def format_logit_weights(row):
    lines = []

    for emo in EMOTIONS:
        col = f"logit_{emo}"

        if col in row.index:
            lines.append(f"{emo}: {row[col]:.4f}")

    return "<br>".join(lines)

# =========================
# Edge stats
# =========================
edges["abs_effect"] = edges["mean_effect"].abs()

if "ablated_target_mean" not in edges.columns:
    edges["ablated_target_mean"] = (
        edges["clean_target_mean"]
        - edges["mean_effect"]
    )

if "normalized_effect" in edges.columns:
    edges["drop_percent"] = (
        edges["normalized_effect"] * 100
    )
else:
    edges["drop_percent"] = (
        edges["mean_effect"]
        / (
            edges["clean_target_mean"].abs()
            + 1e-8
        )
        * 100
    )

edges["abs_drop_percent"] = (
    edges["drop_percent"].abs()
)

print(f"Nodes: {len(features)}")
print(f"Edges: {len(edges)}")
print("Shared nodes:", int(features["is_shared"].sum()))

# =========================
# Layout
# =========================
features = features.sort_values(
    ["layer_idx", "feature_idx"]
).reset_index(drop=True)

emotion_y_base = {
    emo: i * EMOTION_BAND_GAP
    for i, emo in enumerate(EMOTIONS)
}

SHARED_SECONDARY_PULL = 0.15
# 0.0 = exactly primary band
# 0.5 = halfway between primary and secondary
# 1.0 = exactly secondary band

def get_y_base(row):
    primary = row["top_logit_emotion"]
    secondary = row["second_logit_emotion"]

    primary_y = emotion_y_base[primary]

    if not row["is_shared"]:
        return primary_y

    secondary_y = emotion_y_base[secondary]

    return (
        primary_y * (1.0 - SHARED_SECONDARY_PULL)
        + secondary_y * SHARED_SECONDARY_PULL
    )


def get_layout_group(row):
    primary = row["top_logit_emotion"]

    if row["is_shared"]:
        secondary = row["second_logit_emotion"]
        return f"{primary}_shared_with_{secondary}"

    return primary


features["layout_group"] = (
    features.apply(get_layout_group, axis=1)
)

features["y_base"] = (
    features.apply(get_y_base, axis=1)
)

features["within_group_idx"] = (
    features.groupby(
        ["layout_group", "layer_idx"]
    ).cumcount()
)

features["group_count"] = (
    features.groupby(
        ["layout_group", "layer_idx"]
    )["node_id"]
    .transform("count")
)

features["y_pos"] = (
    features["y_base"]
    + features["within_group_idx"]
    - (features["group_count"] - 1) / 2
)

np.random.seed(RANDOM_SEED)

features["x_pos"] = (
    features["layer_idx"]
    + np.random.uniform(-0.06, 0.06, len(features))
)

features["y_pos"] = (
    features["y_pos"]
    + np.random.uniform(-0.06, 0.06, len(features))
)

# =========================
# Node sizes
# =========================
v = features["max_abs_logit"].to_numpy()

v = np.clip(v, 1e-6, None)

v_log = np.log(v)

v_norm = (
    (v_log - v_log.min())
    / (v_log.max() - v_log.min() + 1e-8)
)

features["node_size"] = (
    NODE_MIN_SIZE
    + v_norm * (NODE_MAX_SIZE - NODE_MIN_SIZE)
)

# =========================
# Node positions
# =========================
pos = {
    row["node_id"]: (
        row["x_pos"],
        row["y_pos"],
    )
    for _, row in features.iterrows()
}

# =========================
# Incoming / outgoing
# =========================
incoming = {
    node_id: []
    for node_id in features["node_id"]
}

outgoing = {
    node_id: []
    for node_id in features["node_id"]
}

for _, row in edges.iterrows():
    s = row["source_node_id"]
    t = row["target_node_id"]

    if s not in pos or t not in pos:
        continue

    effect = row["mean_effect"]

    clean = row["clean_target_mean"]
    ablated = row["ablated_target_mean"]

    drop = row["drop_percent"]

    outgoing[s].append(
        (
            t,
            row["target_layer"],
            effect,
            clean,
            ablated,
            drop,
        )
    )

    incoming[t].append(
        (
            s,
            row["source_layer"],
            effect,
            clean,
            ablated,
            drop,
        )
    )

for k in incoming:
    incoming[k].sort(key=lambda x: x[1])

for k in outgoing:
    outgoing[k].sort(key=lambda x: x[1])

# =========================
# Hover formatting
# =========================
def format_edges(edges_list):
    if len(edges_list) == 0:
        return "None"

    lines = []

    for other, layer, eff, clean, ablated, drop in edges_list:
        sign = "+" if eff >= 0 else ""

        lines.append(
            f"{other} (L{layer}): "
            f"clean {clean:.3f} → {ablated:.3f}, "
            f"Δ {sign}{eff:.3f}, "
            f"drop {drop:.1f}%"
        )

    return "<br>".join(lines)

# =========================
# Edge traces
# =========================
edge_traces = []

max_drop = (
    edges["abs_drop_percent"].max()
    if len(edges) > 0
    else 1.0
)

for _, row in edges.iterrows():
    s = row["source_node_id"]
    t = row["target_node_id"]

    if s not in pos or t not in pos:
        continue

    x0, y0 = pos[s]
    x1, y1 = pos[t]

    drop = abs(row["drop_percent"])
    effect = row["mean_effect"]

    width = (
        0.4
        + 4.0 * (drop / (max_drop + 1e-8))
    )

    dash = "solid" if effect >= 0 else "dot"

    color = EMOTION_COLORS.get(
        row.get("conditioning_emotion", ""),
        "#777",
    )

    edge_traces.append(
        go.Scatter(
            x=[x0, x1],
            y=[y0, y1],
            mode="lines",
            line=dict(
                width=width,
                dash=dash,
                color=color,
            ),
            opacity=EDGE_OPACITY,
            hoverinfo="text",
            text=(
                f"{s} → {t}<br>"
                f"Conditioning emotion: "
                f"{row.get('conditioning_emotion', '')}<br>"
                f"clean {row['clean_target_mean']:.3f} "
                f"→ {row['ablated_target_mean']:.3f}<br>"
                f"Δ {row['mean_effect']:.3f}, "
                f"drop {row['drop_percent']:.2f}%"
            ),
            showlegend=False,
        )
    )

# =========================
# Single-emotion nodes
# =========================
node_traces = []

for emotion in EMOTIONS:
    sub = features[
        (features["num_affected_emotions"] == 1)
        & (
            features["affected_emotion_list"]
            .apply(lambda xs: emotion in xs)
        )
    ]

    if len(sub) == 0:
        continue

    hover = []

    for _, row in sub.iterrows():
        node_id = row["node_id"]

        desc = "<br>".join(
            textwrap.wrap(
                str(row.get("description", "")),
                width=80,
            )
        )

        logit_weights = format_logit_weights(row)

        hover.append(
            f"<b>{node_id}</b><br>"
            f"Layer: {row['layer_idx']}<br>"
            f"Feature: {row['feature_idx']}<br>"
            f"<br><b>Affected emotions:</b> "
            f"{row['affected_emotions_clean']}<br>"
            f"<br><b>Shared node:</b> No<br>"
            f"<br><b>Logit weights:</b><br>"
            f"{logit_weights}<br>"
            f"<br><b>Description:</b><br>"
            f"{desc}<br>"
            f"<br><b>Incoming:</b><br>"
            f"{format_edges(incoming[node_id])}<br>"
            f"<br><b>Outgoing:</b><br>"
            f"{format_edges(outgoing[node_id])}"
        )

    node_traces.append(
        go.Scatter(
            x=sub["x_pos"],
            y=sub["y_pos"],
            mode="markers+text",
            text=sub["node_id"],
            textposition="middle center",
            textfont=dict(
                size=7,
                color="white",
            ),
            hoverinfo="text",
            hovertext=hover,
            marker=dict(
                size=sub["node_size"],
                color=EMOTION_COLORS[emotion],
                line=dict(
                    width=1,
                    color="black",
                ),
            ),
            name=emotion,
        )
    )

# =========================
# Shared nodes
# =========================
shared_nodes = features[
    features["is_shared"]
].copy()

shared_hover = []

for _, row in shared_nodes.iterrows():
    node_id = row["node_id"]

    desc = "<br>".join(
        textwrap.wrap(
            str(row.get("description", "")),
            width=80,
        )
    )

    logit_weights = format_logit_weights(row)

    shared_hover.append(
        f"<b>{node_id} (SHARED)</b><br>"
        f"Layer: {row['layer_idx']}<br>"
        f"Feature: {row['feature_idx']}<br>"
        f"<br><b>Affected emotions:</b> "
        f"{row['affected_emotions_clean']}<br>"
        f"<b># affected emotions:</b> "
        f"{row['num_affected_emotions']}<br>"
        f"<br><b>Primary emotion:</b> "
        f"{row['top_logit_emotion']}<br>"
        f"<b>Border emotion:</b> "
        f"{row['second_logit_emotion']}<br>"
        f"<br><b>Logit weights:</b><br>"
        f"{logit_weights}<br>"
        f"<br><b>Description:</b><br>"
        f"{desc}<br>"
        f"<br><b>Incoming:</b><br>"
        f"{format_edges(incoming[node_id])}<br>"
        f"<br><b>Outgoing:</b><br>"
        f"{format_edges(outgoing[node_id])}"
    )

shared_node_trace = go.Scatter(
    x=shared_nodes["x_pos"],
    y=shared_nodes["y_pos"],
    mode="markers+text",
    text=shared_nodes["node_id"],
    textposition="middle center",
    textfont=dict(
        size=7,
        color="white",
    ),
    hoverinfo="text",
    hovertext=shared_hover,
    marker=dict(
        size=shared_nodes["node_size"],
        color=shared_nodes["top_logit_emotion"]
        .map(EMOTION_COLORS),
        line=dict(
            width=3,
            color=shared_nodes["second_logit_emotion"]
            .map(EMOTION_COLORS),
        ),
    ),
    name="shared features",
    showlegend=True,
)

# =========================
# Emotion labels
# =========================
annotations = []

for emotion in EMOTIONS:
    annotations.append(
        dict(
            x=1.02,
            y=emotion_y_base[emotion],
            xref="paper",
            yref="y",
            text=f"<b>{emotion}</b>",
            showarrow=False,
            font=dict(
                size=14,
                color=EMOTION_COLORS[emotion],
            ),
            xanchor="left",
        )
    )

annotations.append(
    dict(
        x=1.02,
        y=np.mean(list(emotion_y_base.values())),
        xref="paper",
        yref="y",
        text=(
            " "
        ),
        showarrow=False,
        font=dict(
            size=11,
            color="black",
        ),
        xanchor="left",
    )
)

# =========================
# Plot
# =========================
fig = go.Figure(
    data=edge_traces
    + node_traces
    + [shared_node_trace]
)

fig.update_layout(
    title=(
        "All Emotion Features (All Edges)<br>"
        "<sup>"
        "Shared nodes use fill color for the strongest "
        "absolute logit emotion and border color for "
        "the second strongest."
        "</sup>"
    ),
    xaxis=dict(title="Layer"),
    yaxis=dict(showticklabels=False),
    width=1600,
    height=1600,
    annotations=annotations,
)

fig.show()

os.makedirs(
    os.path.dirname(OUTPUT_HTML),
    exist_ok=True,
)

fig.write_html(OUTPUT_HTML)

print("Saved to:", OUTPUT_HTML)

# Gemma-2-9b

In [ ]:
from sae_emotion.emotion_entropy_utils import run_entropy_classification_pipeline


INPUT_PATH = "data/google-gemma-2-9b-feature-act/core_emotion_features_filled_logits_signed.csv"

OUTPUT_PATH = "data/google-gemma-2-9b-feature-act/core_emotion_features_filled_logits_signed_entropy.csv"

PLOT_DIR = "data/google-gemma-2-2b-feature-act/visuals"


df, threshold = run_entropy_classification_pipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    entropy_threshold=None,  # Set to 0.7204 if you want fixed threshold
    plot_dir=PLOT_DIR,
    show_plots=True,
)

print("Final entropy threshold:", threshold)

In [ ]:
import os

CACHE_ROOT = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_ROOT}/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_ROOT}/transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{CACHE_ROOT}/hub"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT

In [ ]:
import torch

from sae_emotion.logit_ablation_utils import (
    model_safe_name,
    run_topk_joint_ablation_pipeline,
    run_topk_selection_pipeline,
)

MODEL_NAME = "google/gemma-2-9b"
SAFE_NAME = model_safe_name(MODEL_NAME)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

FEATURE_CSV_PATH = (
    "data/google-gemma-2-9b-feature-act/"
    "core_emotion_features_filled_logits_signed_entropy.csv"
)

DATASET_CSV_PATH = f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"

JOINT_OUT_PATH = (
    "data/google-gemma-2-9b-feature-act/"
    "topk_joint_ablation_logit_drop.csv"
)

SELECTED_OUT_PATH = (
    "data/google-gemma-2-9b-feature-act/"
    "auto_selected_emotion_specific_topk_features.csv"
)

UNIQUE_OUT_PATH = (
    "data/google-gemma-2-9b-feature-act/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

# 1. Run top-k joint ablation
joint_df = run_topk_joint_ablation_pipeline(
    model_name=MODEL_NAME,
    feature_csv_path=FEATURE_CSV_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_csv_path=JOINT_OUT_PATH,
    emotions=EMOTIONS,
    top_k_list=[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20],
    cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub")),
    sample_size_per_emotion=300,
    batch_size=128,
    apply_prompt_template=False,
    use_abs_ranking=False,
    device="cuda",
    dtype=torch.bfloat16,
)

# 2. Select minimal top-k core features
selected_df, unique_df = run_topk_selection_pipeline(
    feature_csv_path=FEATURE_CSV_PATH,
    joint_ablation_csv_path=JOINT_OUT_PATH,
    selected_output_csv_path=SELECTED_OUT_PATH,
    unique_output_csv_path=UNIQUE_OUT_PATH,
    emotions=EMOTIONS,
    min_signed_drop=0.1,
    coverage_ratio=0.80,
)

In [ ]:
import torch
import pandas as pd

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_examples_by_emotion,
)

from sae_emotion.emotion_graph_utils import (
    run_emotion_feature_graph_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-2-9b"

EMOTION_SPEC_PATH = (
    "data/google-gemma-2-9b-feature-act/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

DATASET_CSV_PATH = (
    "data/combined_zeroshot_filtered_google-gemma-2-9b.csv"
)

OUTPUT_DIR = (
    "data/google-gemma-2-9b-feature-act/"
    "emotion_feature_edges"
)

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

DEVICE = "cuda"
DTYPE = torch.bfloat16

CACHE_DIR = os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub"))

# ============================================================
# Run graph building
# ============================================================

all_edges_df = run_emotion_feature_graph_pipeline(
    model_name=MODEL_NAME,

    emotion_spec_csv_path=EMOTION_SPEC_PATH,

    dataset_csv_path=DATASET_CSV_PATH,

    output_dir=OUTPUT_DIR,

    emotions=EMOTIONS,

    # dataset
    text_col="input_text",
    label_col="emotion",
    sample_size_per_emotion=300,
    random_state=42,
    apply_prompt_template=False,

    # model
    device=DEVICE,
    dtype=DTYPE,
    cache_dir=CACHE_DIR,

    # graph params
    min_source_target_layer_gap=1,

    # edge effect
    use_abs_effect=False,

    # optional conditioning
    condition_on_target_active=False,
    target_active_threshold=0.0,
    min_target_active_count=10,
)

print(all_edges_df.shape)
print(all_edges_df.head())

In [ ]:
import os
import textwrap
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================
# Paths / config
# =========================
FEATURE_PATH = "data/google-gemma-2-9b-feature-act/auto_selected_unique_features_with_affected_emotions.csv"

ALL_EDGE_PATH = (
    "data/google-gemma-2-9b-feature-act/"
    "emotion_feature_edges/all_affected_feature_edges.csv"
)

OUTPUT_HTML = (
    "data/google-gemma-2-9b-feature-act/"
    "visuals/core_features.html"
)

EMOTIONS = [
    "joy",
    "surprise",
    "disgust",
    "anger",
    "sadness",
    "fear",
]

EMOTION_COLORS = {
    "joy": "#F58518",
    "surprise": "#72B7B2",
    "disgust": "#54A24B",
    "anger": "#E45756",
    "sadness": "#4C78A8",
    "fear": "#B279A2",
}

NODE_MIN_SIZE = 18
NODE_MAX_SIZE = 48

EDGE_OPACITY = 0.18

RANDOM_SEED = 42
EMOTION_BAND_GAP = 14.0

# =========================
# Load data
# =========================
features = pd.read_csv(FEATURE_PATH)
edges = pd.read_csv(ALL_EDGE_PATH)

# =========================
# Parse affected emotions
# =========================
def parse_affected_emotions(x):
    if pd.isna(x):
        return []

    x = str(x).strip()

    x = (
        x.replace("[", "")
         .replace("]", "")
         .replace("'", "")
         .replace('"', "")
    )

    if x == "":
        return []

    return [
        e.strip()
        for e in x.split(",")
        if e.strip() in EMOTIONS
    ]


features = features[
    features["affected_emotions"].notna()
].copy()

features["feature_idx"] = features["feature_idx"].astype(int)
features["layer_idx"] = features["layer_idx"].astype(int)

features["node_id"] = (
    "L" + features["layer_idx"].astype(str)
    + "-F" + features["feature_idx"].astype(str)
)

features["affected_emotion_list"] = (
    features["affected_emotions"]
    .apply(parse_affected_emotions)
)

features["affected_emotion_list"] = (
    features["affected_emotion_list"]
    .apply(lambda xs: [e for e in EMOTIONS if e in xs])
)

features["num_affected_emotions"] = (
    features["affected_emotion_list"]
    .apply(len)
)

features["is_shared"] = (
    features["num_affected_emotions"] >= 2
)

features["affected_emotions_clean"] = (
    features["affected_emotion_list"]
    .apply(lambda x: ", ".join(x))
)

# =========================
# Edge node IDs
# =========================
edges["source_feature"] = edges["source_feature"].astype(int)
edges["target_feature"] = edges["target_feature"].astype(int)

edges["source_layer"] = edges["source_layer"].astype(int)
edges["target_layer"] = edges["target_layer"].astype(int)

edges["source_node_id"] = (
    "L" + edges["source_layer"].astype(str)
    + "-F" + edges["source_feature"].astype(str)
)

edges["target_node_id"] = (
    "L" + edges["target_layer"].astype(str)
    + "-F" + edges["target_feature"].astype(str)
)

valid_nodes = set(features["node_id"])

edges = edges[
    edges["source_node_id"].isin(valid_nodes)
    & edges["target_node_id"].isin(valid_nodes)
].copy()

# =========================
# Logit columns
# =========================
LOGIT_COLS = [f"logit_{e}" for e in EMOTIONS]

existing_logit_cols = [
    c for c in LOGIT_COLS
    if c in features.columns
]

if len(existing_logit_cols) == 0:
    raise ValueError("No logit_* columns found.")

if "max_abs_logit" not in features.columns:
    features["max_abs_logit"] = (
        features[existing_logit_cols]
        .abs()
        .max(axis=1)
    )

# =========================
# Top / second emotion
# =========================
def get_top_two_logit_emotions(row):
    vals = []

    for emo in EMOTIONS:
        col = f"logit_{emo}"

        if col in row.index:
            vals.append((emo, abs(row[col])))

    vals = sorted(vals, key=lambda x: x[1], reverse=True)

    top_emo = vals[0][0]

    if len(vals) > 1:
        second_emo = vals[1][0]
    else:
        second_emo = top_emo

    return top_emo, second_emo


features[["top_logit_emotion", "second_logit_emotion"]] = (
    features.apply(
        lambda row: pd.Series(
            get_top_two_logit_emotions(row)
        ),
        axis=1,
    )
)

# =========================
# Format logit weights
# =========================
def format_logit_weights(row):
    lines = []

    for emo in EMOTIONS:
        col = f"logit_{emo}"

        if col in row.index:
            lines.append(f"{emo}: {row[col]:.4f}")

    return "<br>".join(lines)

# =========================
# Edge stats
# =========================
edges["abs_effect"] = edges["mean_effect"].abs()

if "ablated_target_mean" not in edges.columns:
    edges["ablated_target_mean"] = (
        edges["clean_target_mean"]
        - edges["mean_effect"]
    )

if "normalized_effect" in edges.columns:
    edges["drop_percent"] = (
        edges["normalized_effect"] * 100
    )
else:
    edges["drop_percent"] = (
        edges["mean_effect"]
        / (
            edges["clean_target_mean"].abs()
            + 1e-8
        )
        * 100
    )

edges["abs_drop_percent"] = (
    edges["drop_percent"].abs()
)

print(f"Nodes: {len(features)}")
print(f"Edges: {len(edges)}")
print("Shared nodes:", int(features["is_shared"].sum()))

# =========================
# Layout
# =========================
features = features.sort_values(
    ["layer_idx", "feature_idx"]
).reset_index(drop=True)

emotion_y_base = {
    emo: i * EMOTION_BAND_GAP
    for i, emo in enumerate(EMOTIONS)
}

SHARED_SECONDARY_PULL = 0.15
# 0.0 = exactly primary band
# 0.5 = halfway between primary and secondary
# 1.0 = exactly secondary band

def get_y_base(row):
    primary = row["top_logit_emotion"]
    secondary = row["second_logit_emotion"]

    primary_y = emotion_y_base[primary]

    if not row["is_shared"]:
        return primary_y

    secondary_y = emotion_y_base[secondary]

    return (
        primary_y * (1.0 - SHARED_SECONDARY_PULL)
        + secondary_y * SHARED_SECONDARY_PULL
    )


def get_layout_group(row):
    primary = row["top_logit_emotion"]

    if row["is_shared"]:
        secondary = row["second_logit_emotion"]
        return f"{primary}_shared_with_{secondary}"

    return primary


features["layout_group"] = (
    features.apply(get_layout_group, axis=1)
)

features["y_base"] = (
    features.apply(get_y_base, axis=1)
)

features["within_group_idx"] = (
    features.groupby(
        ["layout_group", "layer_idx"]
    ).cumcount()
)

features["group_count"] = (
    features.groupby(
        ["layout_group", "layer_idx"]
    )["node_id"]
    .transform("count")
)

features["y_pos"] = (
    features["y_base"]
    + features["within_group_idx"]
    - (features["group_count"] - 1) / 2
)

np.random.seed(RANDOM_SEED)

features["x_pos"] = (
    features["layer_idx"]
    + np.random.uniform(-0.06, 0.06, len(features))
)

features["y_pos"] = (
    features["y_pos"]
    + np.random.uniform(-0.06, 0.06, len(features))
)

# =========================
# Node sizes
# =========================
v = features["max_abs_logit"].to_numpy()

v = np.clip(v, 1e-6, None)

v_log = np.log(v)

v_norm = (
    (v_log - v_log.min())
    / (v_log.max() - v_log.min() + 1e-8)
)

features["node_size"] = (
    NODE_MIN_SIZE
    + v_norm * (NODE_MAX_SIZE - NODE_MIN_SIZE)
)

# =========================
# Node positions
# =========================
pos = {
    row["node_id"]: (
        row["x_pos"],
        row["y_pos"],
    )
    for _, row in features.iterrows()
}

# =========================
# Incoming / outgoing
# =========================
incoming = {
    node_id: []
    for node_id in features["node_id"]
}

outgoing = {
    node_id: []
    for node_id in features["node_id"]
}

for _, row in edges.iterrows():
    s = row["source_node_id"]
    t = row["target_node_id"]

    if s not in pos or t not in pos:
        continue

    effect = row["mean_effect"]

    clean = row["clean_target_mean"]
    ablated = row["ablated_target_mean"]

    drop = row["drop_percent"]

    outgoing[s].append(
        (
            t,
            row["target_layer"],
            effect,
            clean,
            ablated,
            drop,
        )
    )

    incoming[t].append(
        (
            s,
            row["source_layer"],
            effect,
            clean,
            ablated,
            drop,
        )
    )

for k in incoming:
    incoming[k].sort(key=lambda x: x[1])

for k in outgoing:
    outgoing[k].sort(key=lambda x: x[1])

# =========================
# Hover formatting
# =========================
def format_edges(edges_list):
    if len(edges_list) == 0:
        return "None"

    lines = []

    for other, layer, eff, clean, ablated, drop in edges_list:
        sign = "+" if eff >= 0 else ""

        lines.append(
            f"{other} (L{layer}): "
            f"clean {clean:.3f} → {ablated:.3f}, "
            f"Δ {sign}{eff:.3f}, "
            f"drop {drop:.1f}%"
        )

    return "<br>".join(lines)

# =========================
# Edge traces
# =========================
edge_traces = []

max_drop = (
    edges["abs_drop_percent"].max()
    if len(edges) > 0
    else 1.0
)

for _, row in edges.iterrows():
    s = row["source_node_id"]
    t = row["target_node_id"]

    if s not in pos or t not in pos:
        continue

    x0, y0 = pos[s]
    x1, y1 = pos[t]

    drop = abs(row["drop_percent"])
    effect = row["mean_effect"]

    width = (
        0.4
        + 4.0 * (drop / (max_drop + 1e-8))
    )

    dash = "solid" if effect >= 0 else "dot"

    color = EMOTION_COLORS.get(
        row.get("conditioning_emotion", ""),
        "#777",
    )

    edge_traces.append(
        go.Scatter(
            x=[x0, x1],
            y=[y0, y1],
            mode="lines",
            line=dict(
                width=width,
                dash=dash,
                color=color,
            ),
            opacity=EDGE_OPACITY,
            hoverinfo="text",
            text=(
                f"{s} → {t}<br>"
                f"Conditioning emotion: "
                f"{row.get('conditioning_emotion', '')}<br>"
                f"clean {row['clean_target_mean']:.3f} "
                f"→ {row['ablated_target_mean']:.3f}<br>"
                f"Δ {row['mean_effect']:.3f}, "
                f"drop {row['drop_percent']:.2f}%"
            ),
            showlegend=False,
        )
    )

# =========================
# Single-emotion nodes
# =========================
node_traces = []

for emotion in EMOTIONS:
    sub = features[
        (features["num_affected_emotions"] == 1)
        & (
            features["affected_emotion_list"]
            .apply(lambda xs: emotion in xs)
        )
    ]

    if len(sub) == 0:
        continue

    hover = []

    for _, row in sub.iterrows():
        node_id = row["node_id"]

        desc = "<br>".join(
            textwrap.wrap(
                str(row.get("description", "")),
                width=80,
            )
        )

        logit_weights = format_logit_weights(row)

        hover.append(
            f"<b>{node_id}</b><br>"
            f"Layer: {row['layer_idx']}<br>"
            f"Feature: {row['feature_idx']}<br>"
            f"<br><b>Affected emotions:</b> "
            f"{row['affected_emotions_clean']}<br>"
            f"<br><b>Shared node:</b> No<br>"
            f"<br><b>Logit weights:</b><br>"
            f"{logit_weights}<br>"
            f"<br><b>Description:</b><br>"
            f"{desc}<br>"
            f"<br><b>Incoming:</b><br>"
            f"{format_edges(incoming[node_id])}<br>"
            f"<br><b>Outgoing:</b><br>"
            f"{format_edges(outgoing[node_id])}"
        )

    node_traces.append(
        go.Scatter(
            x=sub["x_pos"],
            y=sub["y_pos"],
            mode="markers+text",
            text=sub["node_id"],
            textposition="middle center",
            textfont=dict(
                size=7,
                color="white",
            ),
            hoverinfo="text",
            hovertext=hover,
            marker=dict(
                size=sub["node_size"],
                color=EMOTION_COLORS[emotion],
                line=dict(
                    width=1,
                    color="black",
                ),
            ),
            name=emotion,
        )
    )

# =========================
# Shared nodes
# =========================
shared_nodes = features[
    features["is_shared"]
].copy()

shared_hover = []

for _, row in shared_nodes.iterrows():
    node_id = row["node_id"]

    desc = "<br>".join(
        textwrap.wrap(
            str(row.get("description", "")),
            width=80,
        )
    )

    logit_weights = format_logit_weights(row)

    shared_hover.append(
        f"<b>{node_id} (SHARED)</b><br>"
        f"Layer: {row['layer_idx']}<br>"
        f"Feature: {row['feature_idx']}<br>"
        f"<br><b>Affected emotions:</b> "
        f"{row['affected_emotions_clean']}<br>"
        f"<b># affected emotions:</b> "
        f"{row['num_affected_emotions']}<br>"
        f"<br><b>Primary emotion:</b> "
        f"{row['top_logit_emotion']}<br>"
        f"<b>Border emotion:</b> "
        f"{row['second_logit_emotion']}<br>"
        f"<br><b>Logit weights:</b><br>"
        f"{logit_weights}<br>"
        f"<br><b>Description:</b><br>"
        f"{desc}<br>"
        f"<br><b>Incoming:</b><br>"
        f"{format_edges(incoming[node_id])}<br>"
        f"<br><b>Outgoing:</b><br>"
        f"{format_edges(outgoing[node_id])}"
    )

shared_node_trace = go.Scatter(
    x=shared_nodes["x_pos"],
    y=shared_nodes["y_pos"],
    mode="markers+text",
    text=shared_nodes["node_id"],
    textposition="middle center",
    textfont=dict(
        size=7,
        color="white",
    ),
    hoverinfo="text",
    hovertext=shared_hover,
    marker=dict(
        size=shared_nodes["node_size"],
        color=shared_nodes["top_logit_emotion"]
        .map(EMOTION_COLORS),
        line=dict(
            width=3,
            color=shared_nodes["second_logit_emotion"]
            .map(EMOTION_COLORS),
        ),
    ),
    name="shared features",
    showlegend=True,
)

# =========================
# Emotion labels
# =========================
annotations = []

for emotion in EMOTIONS:
    annotations.append(
        dict(
            x=1.02,
            y=emotion_y_base[emotion],
            xref="paper",
            yref="y",
            text=f"<b>{emotion}</b>",
            showarrow=False,
            font=dict(
                size=14,
                color=EMOTION_COLORS[emotion],
            ),
            xanchor="left",
        )
    )

annotations.append(
    dict(
        x=1.02,
        y=np.mean(list(emotion_y_base.values())),
        xref="paper",
        yref="y",
        text=(
            " "
        ),
        showarrow=False,
        font=dict(
            size=11,
            color="black",
        ),
        xanchor="left",
    )
)

# =========================
# Plot
# =========================
fig = go.Figure(
    data=edge_traces
    + node_traces
    + [shared_node_trace]
)

fig.update_layout(
    title=(
        "All Emotion Features (All Edges)<br>"
        "<sup>"
        "Shared nodes use fill color for the strongest "
        "absolute logit emotion and border color for "
        "the second strongest."
        "</sup>"
    ),
    xaxis=dict(title="Layer"),
    yaxis=dict(showticklabels=False),
    width=1600,
    height=1600,
    annotations=annotations,
)

fig.show()

os.makedirs(
    os.path.dirname(OUTPUT_HTML),
    exist_ok=True,
)

fig.write_html(OUTPUT_HTML)

print("Saved to:", OUTPUT_HTML)

# Llama-3-8B

In [ ]:
from sae_emotion.emotion_entropy_utils import run_entropy_classification_pipeline


INPUT_PATH = "data/meta-llama-Llama-3.1-8B-feature-act/core_emotion_features_filled_logits_signed.csv"

OUTPUT_PATH = "data/meta-llama-Llama-3.1-8B-feature-act/core_emotion_features_filled_logits_signed_entropy.csv"

PLOT_DIR = "data/meta-llama-Llama-3.1-8B-feature-act/visuals"


df, threshold = run_entropy_classification_pipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    entropy_threshold=None,  # Set to 0.7204 if you want fixed threshold
    plot_dir=PLOT_DIR,
    show_plots=True,
)

print("Final entropy threshold:", threshold)

In [ ]:
import torch

from sae_emotion.logit_ablation_utils import (
    model_safe_name,
    run_topk_joint_ablation_pipeline,
    run_topk_selection_pipeline,
)

MODEL_NAME = "meta-llama/Llama-3.1-8B"
SAFE_NAME = model_safe_name(MODEL_NAME)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

FEATURE_CSV_PATH = (
    "data/meta-llama-Llama-3.1-8B-feature-act/"
    "core_emotion_features_filled_logits_signed_entropy.csv"
)

DATASET_CSV_PATH = f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"

JOINT_OUT_PATH = (
    "data/meta-llama-Llama-3.1-8B-feature-act/"
    "topk_joint_ablation_logit_drop.csv"
)

SELECTED_OUT_PATH = (
    "data/meta-llama-Llama-3.1-8B-feature-act/"
    "auto_selected_emotion_specific_topk_features.csv"
)

UNIQUE_OUT_PATH = (
    "data/meta-llama-Llama-3.1-8B-feature-act/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

# 1. Run top-k joint ablation
joint_df = run_topk_joint_ablation_pipeline(
    model_name=MODEL_NAME,
    feature_csv_path=FEATURE_CSV_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_csv_path=JOINT_OUT_PATH,
    emotions=EMOTIONS,
    top_k_list=[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20],
    cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub")),
    sample_size_per_emotion=300,
    batch_size=128,
    apply_prompt_template=False,
    use_abs_ranking=False,
    device="cuda",
    dtype=torch.bfloat16,
)

# 2. Select minimal top-k core features
selected_df, unique_df = run_topk_selection_pipeline(
    feature_csv_path=FEATURE_CSV_PATH,
    joint_ablation_csv_path=JOINT_OUT_PATH,
    selected_output_csv_path=SELECTED_OUT_PATH,
    unique_output_csv_path=UNIQUE_OUT_PATH,
    emotions=EMOTIONS,
    min_signed_drop=0.1,
    coverage_ratio=0.80,
)

In [ ]:
import torch
import pandas as pd

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_examples_by_emotion,
)

from sae_emotion.emotion_graph_utils import (
    run_emotion_feature_graph_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "meta-llama/Llama-3.1-8B"

EMOTION_SPEC_PATH = (
    "data/meta-llama-Llama-3.1-8B-feature-act/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

DATASET_CSV_PATH = (
    "data/combined_zeroshot_filtered_meta-llama-Llama-3.1-8B.csv"
)

OUTPUT_DIR = (
    "data/meta-llama-Llama-3.1-8B-feature-act/"
    "emotion_feature_edges"
)

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

DEVICE = "cuda"
DTYPE = torch.bfloat16

CACHE_DIR = os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub"))

# ============================================================
# Run graph building
# ============================================================

all_edges_df = run_emotion_feature_graph_pipeline(
    model_name=MODEL_NAME,

    emotion_spec_csv_path=EMOTION_SPEC_PATH,

    dataset_csv_path=DATASET_CSV_PATH,

    output_dir=OUTPUT_DIR,

    emotions=EMOTIONS,

    # dataset
    text_col="input_text",
    label_col="emotion",
    sample_size_per_emotion=300,
    random_state=42,
    apply_prompt_template=False,

    # model
    device=DEVICE,
    dtype=DTYPE,
    cache_dir=CACHE_DIR,

    # graph params
    min_source_target_layer_gap=1,

    # edge effect
    use_abs_effect=False,

    # optional conditioning
    condition_on_target_active=False,
    target_active_threshold=0.0,
    min_target_active_count=10,
)

print(all_edges_df.shape)
print(all_edges_df.head())

In [ ]:
import os
import textwrap
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================
# Paths / config
# =========================
FEATURE_PATH = "data/meta-llama-Llama-3.1-8B-feature-act/auto_selected_unique_features_with_affected_emotions.csv"

ALL_EDGE_PATH = (
    "data/meta-llama-Llama-3.1-8B-feature-act/"
    "emotion_feature_edges/all_affected_feature_edges.csv"
)

OUTPUT_HTML = (
    "data/meta-llama-Llama-3.1-8B-feature-act/"
    "visuals/core_features.html"
)

EMOTIONS = [
    "joy",
    "surprise",
    "disgust",
    "anger",
    "sadness",
    "fear",
]

EMOTION_COLORS = {
    "joy": "#F58518",
    "surprise": "#72B7B2",
    "disgust": "#54A24B",
    "anger": "#E45756",
    "sadness": "#4C78A8",
    "fear": "#B279A2",
}

NODE_MIN_SIZE = 18
NODE_MAX_SIZE = 48

EDGE_OPACITY = 0.18

RANDOM_SEED = 42
EMOTION_BAND_GAP = 14.0

# =========================
# Load data
# =========================
features = pd.read_csv(FEATURE_PATH)
edges = pd.read_csv(ALL_EDGE_PATH)

# =========================
# Parse affected emotions
# =========================
def parse_affected_emotions(x):
    if pd.isna(x):
        return []

    x = str(x).strip()

    x = (
        x.replace("[", "")
         .replace("]", "")
         .replace("'", "")
         .replace('"', "")
    )

    if x == "":
        return []

    return [
        e.strip()
        for e in x.split(",")
        if e.strip() in EMOTIONS
    ]


features = features[
    features["affected_emotions"].notna()
].copy()

features["feature_idx"] = features["feature_idx"].astype(int)
features["layer_idx"] = features["layer_idx"].astype(int)

features["node_id"] = (
    "L" + features["layer_idx"].astype(str)
    + "-F" + features["feature_idx"].astype(str)
)

features["affected_emotion_list"] = (
    features["affected_emotions"]
    .apply(parse_affected_emotions)
)

features["affected_emotion_list"] = (
    features["affected_emotion_list"]
    .apply(lambda xs: [e for e in EMOTIONS if e in xs])
)

features["num_affected_emotions"] = (
    features["affected_emotion_list"]
    .apply(len)
)

features["is_shared"] = (
    features["num_affected_emotions"] >= 2
)

features["affected_emotions_clean"] = (
    features["affected_emotion_list"]
    .apply(lambda x: ", ".join(x))
)

# =========================
# Edge node IDs
# =========================
edges["source_feature"] = edges["source_feature"].astype(int)
edges["target_feature"] = edges["target_feature"].astype(int)

edges["source_layer"] = edges["source_layer"].astype(int)
edges["target_layer"] = edges["target_layer"].astype(int)

edges["source_node_id"] = (
    "L" + edges["source_layer"].astype(str)
    + "-F" + edges["source_feature"].astype(str)
)

edges["target_node_id"] = (
    "L" + edges["target_layer"].astype(str)
    + "-F" + edges["target_feature"].astype(str)
)

valid_nodes = set(features["node_id"])

edges = edges[
    edges["source_node_id"].isin(valid_nodes)
    & edges["target_node_id"].isin(valid_nodes)
].copy()

# =========================
# Logit columns
# =========================
LOGIT_COLS = [f"logit_{e}" for e in EMOTIONS]

existing_logit_cols = [
    c for c in LOGIT_COLS
    if c in features.columns
]

if len(existing_logit_cols) == 0:
    raise ValueError("No logit_* columns found.")

if "max_abs_logit" not in features.columns:
    features["max_abs_logit"] = (
        features[existing_logit_cols]
        .abs()
        .max(axis=1)
    )

# =========================
# Top / second emotion
# =========================
def get_top_two_logit_emotions(row):
    vals = []

    for emo in EMOTIONS:
        col = f"logit_{emo}"

        if col in row.index:
            vals.append((emo, abs(row[col])))

    vals = sorted(vals, key=lambda x: x[1], reverse=True)

    top_emo = vals[0][0]

    if len(vals) > 1:
        second_emo = vals[1][0]
    else:
        second_emo = top_emo

    return top_emo, second_emo


features[["top_logit_emotion", "second_logit_emotion"]] = (
    features.apply(
        lambda row: pd.Series(
            get_top_two_logit_emotions(row)
        ),
        axis=1,
    )
)

# =========================
# Format logit weights
# =========================
def format_logit_weights(row):
    lines = []

    for emo in EMOTIONS:
        col = f"logit_{emo}"

        if col in row.index:
            lines.append(f"{emo}: {row[col]:.4f}")

    return "<br>".join(lines)

# =========================
# Edge stats
# =========================
edges["abs_effect"] = edges["mean_effect"].abs()

if "ablated_target_mean" not in edges.columns:
    edges["ablated_target_mean"] = (
        edges["clean_target_mean"]
        - edges["mean_effect"]
    )

if "normalized_effect" in edges.columns:
    edges["drop_percent"] = (
        edges["normalized_effect"] * 100
    )
else:
    edges["drop_percent"] = (
        edges["mean_effect"]
        / (
            edges["clean_target_mean"].abs()
            + 1e-8
        )
        * 100
    )

edges["abs_drop_percent"] = (
    edges["drop_percent"].abs()
)

print(f"Nodes: {len(features)}")
print(f"Edges: {len(edges)}")
print("Shared nodes:", int(features["is_shared"].sum()))

# =========================
# Layout
# =========================
features = features.sort_values(
    ["layer_idx", "feature_idx"]
).reset_index(drop=True)

emotion_y_base = {
    emo: i * EMOTION_BAND_GAP
    for i, emo in enumerate(EMOTIONS)
}

SHARED_SECONDARY_PULL = 0.15
# 0.0 = exactly primary band
# 0.5 = halfway between primary and secondary
# 1.0 = exactly secondary band

def get_y_base(row):
    primary = row["top_logit_emotion"]
    secondary = row["second_logit_emotion"]

    primary_y = emotion_y_base[primary]

    if not row["is_shared"]:
        return primary_y

    secondary_y = emotion_y_base[secondary]

    return (
        primary_y * (1.0 - SHARED_SECONDARY_PULL)
        + secondary_y * SHARED_SECONDARY_PULL
    )


def get_layout_group(row):
    primary = row["top_logit_emotion"]

    if row["is_shared"]:
        secondary = row["second_logit_emotion"]
        return f"{primary}_shared_with_{secondary}"

    return primary


features["layout_group"] = (
    features.apply(get_layout_group, axis=1)
)

features["y_base"] = (
    features.apply(get_y_base, axis=1)
)

features["within_group_idx"] = (
    features.groupby(
        ["layout_group", "layer_idx"]
    ).cumcount()
)

features["group_count"] = (
    features.groupby(
        ["layout_group", "layer_idx"]
    )["node_id"]
    .transform("count")
)

features["y_pos"] = (
    features["y_base"]
    + features["within_group_idx"]
    - (features["group_count"] - 1) / 2
)

np.random.seed(RANDOM_SEED)

features["x_pos"] = (
    features["layer_idx"]
    + np.random.uniform(-0.06, 0.06, len(features))
)

features["y_pos"] = (
    features["y_pos"]
    + np.random.uniform(-0.06, 0.06, len(features))
)

# =========================
# Node sizes
# =========================
v = features["max_abs_logit"].to_numpy()

v = np.clip(v, 1e-6, None)

v_log = np.log(v)

v_norm = (
    (v_log - v_log.min())
    / (v_log.max() - v_log.min() + 1e-8)
)

features["node_size"] = (
    NODE_MIN_SIZE
    + v_norm * (NODE_MAX_SIZE - NODE_MIN_SIZE)
)

# =========================
# Node positions
# =========================
pos = {
    row["node_id"]: (
        row["x_pos"],
        row["y_pos"],
    )
    for _, row in features.iterrows()
}

# =========================
# Incoming / outgoing
# =========================
incoming = {
    node_id: []
    for node_id in features["node_id"]
}

outgoing = {
    node_id: []
    for node_id in features["node_id"]
}

for _, row in edges.iterrows():
    s = row["source_node_id"]
    t = row["target_node_id"]

    if s not in pos or t not in pos:
        continue

    effect = row["mean_effect"]

    clean = row["clean_target_mean"]
    ablated = row["ablated_target_mean"]

    drop = row["drop_percent"]

    outgoing[s].append(
        (
            t,
            row["target_layer"],
            effect,
            clean,
            ablated,
            drop,
        )
    )

    incoming[t].append(
        (
            s,
            row["source_layer"],
            effect,
            clean,
            ablated,
            drop,
        )
    )

for k in incoming:
    incoming[k].sort(key=lambda x: x[1])

for k in outgoing:
    outgoing[k].sort(key=lambda x: x[1])

# =========================
# Hover formatting
# =========================
def format_edges(edges_list):
    if len(edges_list) == 0:
        return "None"

    lines = []

    for other, layer, eff, clean, ablated, drop in edges_list:
        sign = "+" if eff >= 0 else ""

        lines.append(
            f"{other} (L{layer}): "
            f"clean {clean:.3f} → {ablated:.3f}, "
            f"Δ {sign}{eff:.3f}, "
            f"drop {drop:.1f}%"
        )

    return "<br>".join(lines)

# =========================
# Edge traces
# =========================
edge_traces = []

max_drop = (
    edges["abs_drop_percent"].max()
    if len(edges) > 0
    else 1.0
)

for _, row in edges.iterrows():
    s = row["source_node_id"]
    t = row["target_node_id"]

    if s not in pos or t not in pos:
        continue

    x0, y0 = pos[s]
    x1, y1 = pos[t]

    drop = abs(row["drop_percent"])
    effect = row["mean_effect"]

    width = (
        0.4
        + 4.0 * (drop / (max_drop + 1e-8))
    )

    dash = "solid" if effect >= 0 else "dot"

    color = EMOTION_COLORS.get(
        row.get("conditioning_emotion", ""),
        "#777",
    )

    edge_traces.append(
        go.Scatter(
            x=[x0, x1],
            y=[y0, y1],
            mode="lines",
            line=dict(
                width=width,
                dash=dash,
                color=color,
            ),
            opacity=EDGE_OPACITY,
            hoverinfo="text",
            text=(
                f"{s} → {t}<br>"
                f"Conditioning emotion: "
                f"{row.get('conditioning_emotion', '')}<br>"
                f"clean {row['clean_target_mean']:.3f} "
                f"→ {row['ablated_target_mean']:.3f}<br>"
                f"Δ {row['mean_effect']:.3f}, "
                f"drop {row['drop_percent']:.2f}%"
            ),
            showlegend=False,
        )
    )

# =========================
# Single-emotion nodes
# =========================
node_traces = []

for emotion in EMOTIONS:
    sub = features[
        (features["num_affected_emotions"] == 1)
        & (
            features["affected_emotion_list"]
            .apply(lambda xs: emotion in xs)
        )
    ]

    if len(sub) == 0:
        continue

    hover = []

    for _, row in sub.iterrows():
        node_id = row["node_id"]

        desc = "<br>".join(
            textwrap.wrap(
                str(row.get("description", "")),
                width=80,
            )
        )

        logit_weights = format_logit_weights(row)

        hover.append(
            f"<b>{node_id}</b><br>"
            f"Layer: {row['layer_idx']}<br>"
            f"Feature: {row['feature_idx']}<br>"
            f"<br><b>Affected emotions:</b> "
            f"{row['affected_emotions_clean']}<br>"
            f"<br><b>Shared node:</b> No<br>"
            f"<br><b>Logit weights:</b><br>"
            f"{logit_weights}<br>"
            f"<br><b>Description:</b><br>"
            f"{desc}<br>"
            f"<br><b>Incoming:</b><br>"
            f"{format_edges(incoming[node_id])}<br>"
            f"<br><b>Outgoing:</b><br>"
            f"{format_edges(outgoing[node_id])}"
        )

    node_traces.append(
        go.Scatter(
            x=sub["x_pos"],
            y=sub["y_pos"],
            mode="markers+text",
            text=sub["node_id"],
            textposition="middle center",
            textfont=dict(
                size=7,
                color="white",
            ),
            hoverinfo="text",
            hovertext=hover,
            marker=dict(
                size=sub["node_size"],
                color=EMOTION_COLORS[emotion],
                line=dict(
                    width=1,
                    color="black",
                ),
            ),
            name=emotion,
        )
    )

# =========================
# Shared nodes
# =========================
shared_nodes = features[
    features["is_shared"]
].copy()

shared_hover = []

for _, row in shared_nodes.iterrows():
    node_id = row["node_id"]

    desc = "<br>".join(
        textwrap.wrap(
            str(row.get("description", "")),
            width=80,
        )
    )

    logit_weights = format_logit_weights(row)

    shared_hover.append(
        f"<b>{node_id} (SHARED)</b><br>"
        f"Layer: {row['layer_idx']}<br>"
        f"Feature: {row['feature_idx']}<br>"
        f"<br><b>Affected emotions:</b> "
        f"{row['affected_emotions_clean']}<br>"
        f"<b># affected emotions:</b> "
        f"{row['num_affected_emotions']}<br>"
        f"<br><b>Primary emotion:</b> "
        f"{row['top_logit_emotion']}<br>"
        f"<b>Border emotion:</b> "
        f"{row['second_logit_emotion']}<br>"
        f"<br><b>Logit weights:</b><br>"
        f"{logit_weights}<br>"
        f"<br><b>Description:</b><br>"
        f"{desc}<br>"
        f"<br><b>Incoming:</b><br>"
        f"{format_edges(incoming[node_id])}<br>"
        f"<br><b>Outgoing:</b><br>"
        f"{format_edges(outgoing[node_id])}"
    )

shared_node_trace = go.Scatter(
    x=shared_nodes["x_pos"],
    y=shared_nodes["y_pos"],
    mode="markers+text",
    text=shared_nodes["node_id"],
    textposition="middle center",
    textfont=dict(
        size=7,
        color="white",
    ),
    hoverinfo="text",
    hovertext=shared_hover,
    marker=dict(
        size=shared_nodes["node_size"],
        color=shared_nodes["top_logit_emotion"]
        .map(EMOTION_COLORS),
        line=dict(
            width=3,
            color=shared_nodes["second_logit_emotion"]
            .map(EMOTION_COLORS),
        ),
    ),
    name="shared features",
    showlegend=True,
)

# =========================
# Emotion labels
# =========================
annotations = []

for emotion in EMOTIONS:
    annotations.append(
        dict(
            x=1.02,
            y=emotion_y_base[emotion],
            xref="paper",
            yref="y",
            text=f"<b>{emotion}</b>",
            showarrow=False,
            font=dict(
                size=14,
                color=EMOTION_COLORS[emotion],
            ),
            xanchor="left",
        )
    )

annotations.append(
    dict(
        x=1.02,
        y=np.mean(list(emotion_y_base.values())),
        xref="paper",
        yref="y",
        text=(
            " "
        ),
        showarrow=False,
        font=dict(
            size=11,
            color="black",
        ),
        xanchor="left",
    )
)

# =========================
# Plot
# =========================
fig = go.Figure(
    data=edge_traces
    + node_traces
    + [shared_node_trace]
)

fig.update_layout(
    title=(
        "All Emotion Features (All Edges)<br>"
        "<sup>"
        "Shared nodes use fill color for the strongest "
        "absolute logit emotion and border color for "
        "the second strongest."
        "</sup>"
    ),
    xaxis=dict(title="Layer"),
    yaxis=dict(showticklabels=False),
    width=1600,
    height=1600,
    annotations=annotations,
)

fig.show()

os.makedirs(
    os.path.dirname(OUTPUT_HTML),
    exist_ok=True,
)

fig.write_html(OUTPUT_HTML)

print("Saved to:", OUTPUT_HTML)

## gemma-3-4b-pt-big

In [ ]:
from sae_emotion.emotion_entropy_utils import run_entropy_classification_pipeline


INPUT_PATH = "data/google-gemma-3-4b-pt-feature-act/core_emotion_features_filled_logits_signed.csv"

OUTPUT_PATH = "data/google-gemma-3-4b-pt-feature-act/core_emotion_features_filled_logits_signed_entropy.csv"

PLOT_DIR = "data/google-gemma-3-4b-pt-feature-act/visuals"


df, threshold = run_entropy_classification_pipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    entropy_threshold=None,  # Set to 0.7204 if you want fixed threshold
    plot_dir=PLOT_DIR,
    show_plots=True,
)

print("Final entropy threshold:", threshold)

In [ ]:
import torch

from sae_emotion.logit_ablation_utils import (
    model_safe_name,
    run_topk_joint_ablation_pipeline,
    run_topk_selection_pipeline,
)

MODEL_NAME = "google/gemma-3-4b-pt"
SAFE_NAME = model_safe_name(MODEL_NAME)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

FEATURE_CSV_PATH = (
    "data/google-gemma-3-4b-pt-feature-act/"
    "core_emotion_features_filled_logits_signed_entropy.csv"
)

DATASET_CSV_PATH = f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"

JOINT_OUT_PATH = (
    "data/google-gemma-3-4b-pt-feature-act/"
    "topk_joint_ablation_logit_drop.csv"
)

SELECTED_OUT_PATH = (
    "data/google-gemma-3-4b-pt-feature-act/"
    "auto_selected_emotion_specific_topk_features.csv"
)

UNIQUE_OUT_PATH = (
    "data/google-gemma-3-4b-pt-feature-act/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

# 1. Run top-k joint ablation
joint_df = run_topk_joint_ablation_pipeline(
    model_name=MODEL_NAME,
    feature_csv_path=FEATURE_CSV_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_csv_path=JOINT_OUT_PATH,
    emotions=EMOTIONS,
    top_k_list=[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20],
    cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub")),
    sample_size_per_emotion=300,
    batch_size=128,
    apply_prompt_template=False,
    use_abs_ranking=False,
    device="cuda",
    dtype=torch.bfloat16,
)

# 2. Select minimal top-k core features
selected_df, unique_df = run_topk_selection_pipeline(
    feature_csv_path=FEATURE_CSV_PATH,
    joint_ablation_csv_path=JOINT_OUT_PATH,
    selected_output_csv_path=SELECTED_OUT_PATH,
    unique_output_csv_path=UNIQUE_OUT_PATH,
    emotions=EMOTIONS,
    min_signed_drop=0.1,
    coverage_ratio=0.80,
)

In [ ]:
import torch
import pandas as pd

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_examples_by_emotion,
)

from sae_emotion.emotion_graph_utils import (
    run_emotion_feature_graph_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-4b-pt"

EMOTION_SPEC_PATH = (
    "data/google-gemma-3-4b-pt-feature-act/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

DATASET_CSV_PATH = (
    "data/combined_zeroshot_filtered_google-gemma-3-4b-pt.csv"
)

OUTPUT_DIR = (
    "data/google-gemma-3-4b-pt-feature-act/"
    "emotion_feature_edges"
)

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

DEVICE = "cuda"
DTYPE = torch.bfloat16

CACHE_DIR = os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub"))

# ============================================================
# Run graph building
# ============================================================

all_edges_df = run_emotion_feature_graph_pipeline(
    model_name=MODEL_NAME,

    emotion_spec_csv_path=EMOTION_SPEC_PATH,

    dataset_csv_path=DATASET_CSV_PATH,

    output_dir=OUTPUT_DIR,

    emotions=EMOTIONS,

    # dataset
    text_col="input_text",
    label_col="emotion",
    sample_size_per_emotion=300,
    random_state=42,
    apply_prompt_template=False,

    # model
    device=DEVICE,
    dtype=DTYPE,
    cache_dir=CACHE_DIR,

    # graph params
    min_source_target_layer_gap=1,

    # edge effect
    use_abs_effect=False,

    # optional conditioning
    condition_on_target_active=False,
    target_active_threshold=0.0,
    min_target_active_count=10,
)

print(all_edges_df.shape)
print(all_edges_df.head())

In [ ]:
import os
import textwrap
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================
# Paths / config
# =========================
FEATURE_PATH = "data/google-gemma-3-4b-pt-feature-act/auto_selected_unique_features_with_affected_emotions.csv"

ALL_EDGE_PATH = (
    "data/google-gemma-3-4b-pt-feature-act/"
    "emotion_feature_edges/all_affected_feature_edges.csv"
)

OUTPUT_HTML = (
    "data/google-gemma-3-4b-pt-feature-act/"
    "visuals/core_features.html"
)

EMOTIONS = [
    "joy",
    "surprise",
    "disgust",
    "anger",
    "sadness",
    "fear",
]

EMOTION_COLORS = {
    "joy": "#F58518",
    "surprise": "#72B7B2",
    "disgust": "#54A24B",
    "anger": "#E45756",
    "sadness": "#4C78A8",
    "fear": "#B279A2",
}

NODE_MIN_SIZE = 18
NODE_MAX_SIZE = 48

EDGE_OPACITY = 0.18

RANDOM_SEED = 42
EMOTION_BAND_GAP = 14.0

# =========================
# Load data
# =========================
features = pd.read_csv(FEATURE_PATH)
edges = pd.read_csv(ALL_EDGE_PATH)

# =========================
# Parse affected emotions
# =========================
def parse_affected_emotions(x):
    if pd.isna(x):
        return []

    x = str(x).strip()

    x = (
        x.replace("[", "")
         .replace("]", "")
         .replace("'", "")
         .replace('"', "")
    )

    if x == "":
        return []

    return [
        e.strip()
        for e in x.split(",")
        if e.strip() in EMOTIONS
    ]


features = features[
    features["affected_emotions"].notna()
].copy()

features["feature_idx"] = features["feature_idx"].astype(int)
features["layer_idx"] = features["layer_idx"].astype(int)

features["node_id"] = (
    "L" + features["layer_idx"].astype(str)
    + "-F" + features["feature_idx"].astype(str)
)

features["affected_emotion_list"] = (
    features["affected_emotions"]
    .apply(parse_affected_emotions)
)

features["affected_emotion_list"] = (
    features["affected_emotion_list"]
    .apply(lambda xs: [e for e in EMOTIONS if e in xs])
)

features["num_affected_emotions"] = (
    features["affected_emotion_list"]
    .apply(len)
)

features["is_shared"] = (
    features["num_affected_emotions"] >= 2
)

features["affected_emotions_clean"] = (
    features["affected_emotion_list"]
    .apply(lambda x: ", ".join(x))
)

# =========================
# Edge node IDs
# =========================
edges["source_feature"] = edges["source_feature"].astype(int)
edges["target_feature"] = edges["target_feature"].astype(int)

edges["source_layer"] = edges["source_layer"].astype(int)
edges["target_layer"] = edges["target_layer"].astype(int)

edges["source_node_id"] = (
    "L" + edges["source_layer"].astype(str)
    + "-F" + edges["source_feature"].astype(str)
)

edges["target_node_id"] = (
    "L" + edges["target_layer"].astype(str)
    + "-F" + edges["target_feature"].astype(str)
)

valid_nodes = set(features["node_id"])

edges = edges[
    edges["source_node_id"].isin(valid_nodes)
    & edges["target_node_id"].isin(valid_nodes)
].copy()

# =========================
# Logit columns
# =========================
LOGIT_COLS = [f"logit_{e}" for e in EMOTIONS]

existing_logit_cols = [
    c for c in LOGIT_COLS
    if c in features.columns
]

if len(existing_logit_cols) == 0:
    raise ValueError("No logit_* columns found.")

if "max_abs_logit" not in features.columns:
    features["max_abs_logit"] = (
        features[existing_logit_cols]
        .abs()
        .max(axis=1)
    )

# =========================
# Top / second emotion
# =========================
def get_top_two_logit_emotions(row):
    vals = []

    for emo in EMOTIONS:
        col = f"logit_{emo}"

        if col in row.index:
            vals.append((emo, abs(row[col])))

    vals = sorted(vals, key=lambda x: x[1], reverse=True)

    top_emo = vals[0][0]

    if len(vals) > 1:
        second_emo = vals[1][0]
    else:
        second_emo = top_emo

    return top_emo, second_emo


features[["top_logit_emotion", "second_logit_emotion"]] = (
    features.apply(
        lambda row: pd.Series(
            get_top_two_logit_emotions(row)
        ),
        axis=1,
    )
)

# =========================
# Format logit weights
# =========================
def format_logit_weights(row):
    lines = []

    for emo in EMOTIONS:
        col = f"logit_{emo}"

        if col in row.index:
            lines.append(f"{emo}: {row[col]:.4f}")

    return "<br>".join(lines)

# =========================
# Edge stats
# =========================
edges["abs_effect"] = edges["mean_effect"].abs()

if "ablated_target_mean" not in edges.columns:
    edges["ablated_target_mean"] = (
        edges["clean_target_mean"]
        - edges["mean_effect"]
    )

if "normalized_effect" in edges.columns:
    edges["drop_percent"] = (
        edges["normalized_effect"] * 100
    )
else:
    edges["drop_percent"] = (
        edges["mean_effect"]
        / (
            edges["clean_target_mean"].abs()
            + 1e-8
        )
        * 100
    )

edges["abs_drop_percent"] = (
    edges["drop_percent"].abs()
)

print(f"Nodes: {len(features)}")
print(f"Edges: {len(edges)}")
print("Shared nodes:", int(features["is_shared"].sum()))

# =========================
# Layout
# =========================
features = features.sort_values(
    ["layer_idx", "feature_idx"]
).reset_index(drop=True)

emotion_y_base = {
    emo: i * EMOTION_BAND_GAP
    for i, emo in enumerate(EMOTIONS)
}

SHARED_SECONDARY_PULL = 0.15
# 0.0 = exactly primary band
# 0.5 = halfway between primary and secondary
# 1.0 = exactly secondary band

def get_y_base(row):
    primary = row["top_logit_emotion"]
    secondary = row["second_logit_emotion"]

    primary_y = emotion_y_base[primary]

    if not row["is_shared"]:
        return primary_y

    secondary_y = emotion_y_base[secondary]

    return (
        primary_y * (1.0 - SHARED_SECONDARY_PULL)
        + secondary_y * SHARED_SECONDARY_PULL
    )


def get_layout_group(row):
    primary = row["top_logit_emotion"]

    if row["is_shared"]:
        secondary = row["second_logit_emotion"]
        return f"{primary}_shared_with_{secondary}"

    return primary


features["layout_group"] = (
    features.apply(get_layout_group, axis=1)
)

features["y_base"] = (
    features.apply(get_y_base, axis=1)
)

features["within_group_idx"] = (
    features.groupby(
        ["layout_group", "layer_idx"]
    ).cumcount()
)

features["group_count"] = (
    features.groupby(
        ["layout_group", "layer_idx"]
    )["node_id"]
    .transform("count")
)

features["y_pos"] = (
    features["y_base"]
    + features["within_group_idx"]
    - (features["group_count"] - 1) / 2
)

np.random.seed(RANDOM_SEED)

features["x_pos"] = (
    features["layer_idx"]
    + np.random.uniform(-0.06, 0.06, len(features))
)

features["y_pos"] = (
    features["y_pos"]
    + np.random.uniform(-0.06, 0.06, len(features))
)

# =========================
# Node sizes
# =========================
v = features["max_abs_logit"].to_numpy()

v = np.clip(v, 1e-6, None)

v_log = np.log(v)

v_norm = (
    (v_log - v_log.min())
    / (v_log.max() - v_log.min() + 1e-8)
)

features["node_size"] = (
    NODE_MIN_SIZE
    + v_norm * (NODE_MAX_SIZE - NODE_MIN_SIZE)
)

# =========================
# Node positions
# =========================
pos = {
    row["node_id"]: (
        row["x_pos"],
        row["y_pos"],
    )
    for _, row in features.iterrows()
}

# =========================
# Incoming / outgoing
# =========================
incoming = {
    node_id: []
    for node_id in features["node_id"]
}

outgoing = {
    node_id: []
    for node_id in features["node_id"]
}

for _, row in edges.iterrows():
    s = row["source_node_id"]
    t = row["target_node_id"]

    if s not in pos or t not in pos:
        continue

    effect = row["mean_effect"]

    clean = row["clean_target_mean"]
    ablated = row["ablated_target_mean"]

    drop = row["drop_percent"]

    outgoing[s].append(
        (
            t,
            row["target_layer"],
            effect,
            clean,
            ablated,
            drop,
        )
    )

    incoming[t].append(
        (
            s,
            row["source_layer"],
            effect,
            clean,
            ablated,
            drop,
        )
    )

for k in incoming:
    incoming[k].sort(key=lambda x: x[1])

for k in outgoing:
    outgoing[k].sort(key=lambda x: x[1])

# =========================
# Hover formatting
# =========================
def format_edges(edges_list):
    if len(edges_list) == 0:
        return "None"

    lines = []

    for other, layer, eff, clean, ablated, drop in edges_list:
        sign = "+" if eff >= 0 else ""

        lines.append(
            f"{other} (L{layer}): "
            f"clean {clean:.3f} → {ablated:.3f}, "
            f"Δ {sign}{eff:.3f}, "
            f"drop {drop:.1f}%"
        )

    return "<br>".join(lines)

# =========================
# Edge traces
# =========================
edge_traces = []

max_drop = (
    edges["abs_drop_percent"].max()
    if len(edges) > 0
    else 1.0
)

for _, row in edges.iterrows():
    s = row["source_node_id"]
    t = row["target_node_id"]

    if s not in pos or t not in pos:
        continue

    x0, y0 = pos[s]
    x1, y1 = pos[t]

    drop = abs(row["drop_percent"])
    effect = row["mean_effect"]

    width = (
        0.4
        + 4.0 * (drop / (max_drop + 1e-8))
    )

    dash = "solid" if effect >= 0 else "dot"

    color = EMOTION_COLORS.get(
        row.get("conditioning_emotion", ""),
        "#777",
    )

    edge_traces.append(
        go.Scatter(
            x=[x0, x1],
            y=[y0, y1],
            mode="lines",
            line=dict(
                width=width,
                dash=dash,
                color=color,
            ),
            opacity=EDGE_OPACITY,
            hoverinfo="text",
            text=(
                f"{s} → {t}<br>"
                f"Conditioning emotion: "
                f"{row.get('conditioning_emotion', '')}<br>"
                f"clean {row['clean_target_mean']:.3f} "
                f"→ {row['ablated_target_mean']:.3f}<br>"
                f"Δ {row['mean_effect']:.3f}, "
                f"drop {row['drop_percent']:.2f}%"
            ),
            showlegend=False,
        )
    )

# =========================
# Single-emotion nodes
# =========================
node_traces = []

for emotion in EMOTIONS:
    sub = features[
        (features["num_affected_emotions"] == 1)
        & (
            features["affected_emotion_list"]
            .apply(lambda xs: emotion in xs)
        )
    ]

    if len(sub) == 0:
        continue

    hover = []

    for _, row in sub.iterrows():
        node_id = row["node_id"]

        desc = "<br>".join(
            textwrap.wrap(
                str(row.get("description", "")),
                width=80,
            )
        )

        logit_weights = format_logit_weights(row)

        hover.append(
            f"<b>{node_id}</b><br>"
            f"Layer: {row['layer_idx']}<br>"
            f"Feature: {row['feature_idx']}<br>"
            f"<br><b>Affected emotions:</b> "
            f"{row['affected_emotions_clean']}<br>"
            f"<br><b>Shared node:</b> No<br>"
            f"<br><b>Logit weights:</b><br>"
            f"{logit_weights}<br>"
            f"<br><b>Description:</b><br>"
            f"{desc}<br>"
            f"<br><b>Incoming:</b><br>"
            f"{format_edges(incoming[node_id])}<br>"
            f"<br><b>Outgoing:</b><br>"
            f"{format_edges(outgoing[node_id])}"
        )

    node_traces.append(
        go.Scatter(
            x=sub["x_pos"],
            y=sub["y_pos"],
            mode="markers+text",
            text=sub["node_id"],
            textposition="middle center",
            textfont=dict(
                size=7,
                color="white",
            ),
            hoverinfo="text",
            hovertext=hover,
            marker=dict(
                size=sub["node_size"],
                color=EMOTION_COLORS[emotion],
                line=dict(
                    width=1,
                    color="black",
                ),
            ),
            name=emotion,
        )
    )

# =========================
# Shared nodes
# =========================
shared_nodes = features[
    features["is_shared"]
].copy()

shared_hover = []

for _, row in shared_nodes.iterrows():
    node_id = row["node_id"]

    desc = "<br>".join(
        textwrap.wrap(
            str(row.get("description", "")),
            width=80,
        )
    )

    logit_weights = format_logit_weights(row)

    shared_hover.append(
        f"<b>{node_id} (SHARED)</b><br>"
        f"Layer: {row['layer_idx']}<br>"
        f"Feature: {row['feature_idx']}<br>"
        f"<br><b>Affected emotions:</b> "
        f"{row['affected_emotions_clean']}<br>"
        f"<b># affected emotions:</b> "
        f"{row['num_affected_emotions']}<br>"
        f"<br><b>Primary emotion:</b> "
        f"{row['top_logit_emotion']}<br>"
        f"<b>Border emotion:</b> "
        f"{row['second_logit_emotion']}<br>"
        f"<br><b>Logit weights:</b><br>"
        f"{logit_weights}<br>"
        f"<br><b>Description:</b><br>"
        f"{desc}<br>"
        f"<br><b>Incoming:</b><br>"
        f"{format_edges(incoming[node_id])}<br>"
        f"<br><b>Outgoing:</b><br>"
        f"{format_edges(outgoing[node_id])}"
    )

shared_node_trace = go.Scatter(
    x=shared_nodes["x_pos"],
    y=shared_nodes["y_pos"],
    mode="markers+text",
    text=shared_nodes["node_id"],
    textposition="middle center",
    textfont=dict(
        size=7,
        color="white",
    ),
    hoverinfo="text",
    hovertext=shared_hover,
    marker=dict(
        size=shared_nodes["node_size"],
        color=shared_nodes["top_logit_emotion"]
        .map(EMOTION_COLORS),
        line=dict(
            width=3,
            color=shared_nodes["second_logit_emotion"]
            .map(EMOTION_COLORS),
        ),
    ),
    name="shared features",
    showlegend=True,
)

# =========================
# Emotion labels
# =========================
annotations = []

for emotion in EMOTIONS:
    annotations.append(
        dict(
            x=1.02,
            y=emotion_y_base[emotion],
            xref="paper",
            yref="y",
            text=f"<b>{emotion}</b>",
            showarrow=False,
            font=dict(
                size=14,
                color=EMOTION_COLORS[emotion],
            ),
            xanchor="left",
        )
    )

annotations.append(
    dict(
        x=1.02,
        y=np.mean(list(emotion_y_base.values())),
        xref="paper",
        yref="y",
        text=(
            " "
        ),
        showarrow=False,
        font=dict(
            size=11,
            color="black",
        ),
        xanchor="left",
    )
)

# =========================
# Plot
# =========================
fig = go.Figure(
    data=edge_traces
    + node_traces
    + [shared_node_trace]
)

fig.update_layout(
    title=(
        "All Emotion Features (All Edges)<br>"
        "<sup>"
        "Shared nodes use fill color for the strongest "
        "absolute logit emotion and border color for "
        "the second strongest."
        "</sup>"
    ),
    xaxis=dict(title="Layer"),
    yaxis=dict(showticklabels=False),
    width=1600,
    height=1600,
    annotations=annotations,
)

fig.show()

os.makedirs(
    os.path.dirname(OUTPUT_HTML),
    exist_ok=True,
)

fig.write_html(OUTPUT_HTML)

print("Saved to:", OUTPUT_HTML)

## gemma-3-4b-pt-small

In [ ]:
from sae_emotion.emotion_entropy_utils import run_entropy_classification_pipeline


INPUT_PATH = "data/google-gemma-3-4b-pt-feature-act-small/core_emotion_features_filled_logits_signed.csv"

OUTPUT_PATH = "data/google-gemma-3-4b-pt-feature-act-small/core_emotion_features_filled_logits_signed_entropy.csv"

PLOT_DIR = "data/google-gemma-3-4b-pt-feature-act-small/visuals"


df, threshold = run_entropy_classification_pipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    entropy_threshold=None,  # Set to 0.7204 if you want fixed threshold
    plot_dir=PLOT_DIR,
    show_plots=True,
)

print("Final entropy threshold:", threshold)

In [ ]:
import torch

from sae_emotion.logit_ablation_utils import (
    model_safe_name,
    run_topk_joint_ablation_pipeline,
    run_topk_selection_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-4b-pt"
SAE_SIZE = "small"   # "big", "medium", or "small"

SAFE_NAME = model_safe_name(MODEL_NAME)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

FEATURE_CSV_PATH = (
    f"{FEATURE_DIR}/core_emotion_features_filled_logits_signed_entropy.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

JOINT_OUT_PATH = (
    f"{FEATURE_DIR}/topk_joint_ablation_logit_drop.csv"
)

SELECTED_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_emotion_specific_topk_features.csv"
)

UNIQUE_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

# ============================================================
# 1. Run top-k joint ablation
# ============================================================

joint_df = run_topk_joint_ablation_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,
    feature_csv_path=FEATURE_CSV_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_csv_path=JOINT_OUT_PATH,
    emotions=EMOTIONS,
    top_k_list=[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20],
    cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub")),
    sample_size_per_emotion=300,
    batch_size=128,
    apply_prompt_template=False,
    use_abs_ranking=False,
    device="cuda",
    dtype=torch.bfloat16,
)

# ============================================================
# 2. Select minimal top-k core features
# ============================================================

selected_df, unique_df = run_topk_selection_pipeline(
    feature_csv_path=FEATURE_CSV_PATH,
    joint_ablation_csv_path=JOINT_OUT_PATH,
    selected_output_csv_path=SELECTED_OUT_PATH,
    unique_output_csv_path=UNIQUE_OUT_PATH,
    emotions=EMOTIONS,
    min_signed_drop=0.1,
    coverage_ratio=0.80,
)

In [ ]:
import torch
import pandas as pd

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_examples_by_emotion,
    model_safe_name,
)

from sae_emotion.emotion_graph_utils import (
    run_emotion_feature_graph_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-4b-pt"
SAE_SIZE = "small"

SAFE_NAME = model_safe_name(MODEL_NAME)
FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

EMOTION_SPEC_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

OUTPUT_DIR = (
    f"{FEATURE_DIR}/emotion_feature_edges"
)

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

DEVICE = "cuda"
DTYPE = torch.bfloat16

CACHE_DIR = None

# ============================================================
# Run graph building
# ============================================================

all_edges_df = run_emotion_feature_graph_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,

    emotion_spec_csv_path=EMOTION_SPEC_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_dir=OUTPUT_DIR,
    emotions=EMOTIONS,

    text_col="input_text",
    label_col="emotion",
    sample_size_per_emotion=300,
    random_state=42,
    apply_prompt_template=False,

    device=DEVICE,
    dtype=DTYPE,
    cache_dir=CACHE_DIR,

    min_source_target_layer_gap=1,

    use_abs_effect=False,

    condition_on_target_active=False,
    target_active_threshold=0.0,
    min_target_active_count=10,
)

print(all_edges_df.shape)
print(all_edges_df.head())

In [ ]:
import os
import textwrap
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================
# Paths / config
# =========================
FEATURE_PATH = "data/google-gemma-3-4b-pt-feature-act-small/auto_selected_unique_features_with_affected_emotions.csv"

ALL_EDGE_PATH = (
    "data/google-gemma-3-4b-pt-feature-act-small/"
    "emotion_feature_edges/all_affected_feature_edges.csv"
)

OUTPUT_HTML = (
    "data/google-gemma-3-4b-pt-feature-act-small/"
    "visuals/core_features.html"
)

EMOTIONS = [
    "joy",
    "surprise",
    "disgust",
    "anger",
    "sadness",
    "fear",
]

EMOTION_COLORS = {
    "joy": "#F58518",
    "surprise": "#72B7B2",
    "disgust": "#54A24B",
    "anger": "#E45756",
    "sadness": "#4C78A8",
    "fear": "#B279A2",
}

NODE_MIN_SIZE = 18
NODE_MAX_SIZE = 48

EDGE_OPACITY = 0.18

RANDOM_SEED = 42
EMOTION_BAND_GAP = 14.0

# =========================
# Load data
# =========================
features = pd.read_csv(FEATURE_PATH)
edges = pd.read_csv(ALL_EDGE_PATH)

# =========================
# Parse affected emotions
# =========================
def parse_affected_emotions(x):
    if pd.isna(x):
        return []

    x = str(x).strip()

    x = (
        x.replace("[", "")
         .replace("]", "")
         .replace("'", "")
         .replace('"', "")
    )

    if x == "":
        return []

    return [
        e.strip()
        for e in x.split(",")
        if e.strip() in EMOTIONS
    ]


features = features[
    features["affected_emotions"].notna()
].copy()

features["feature_idx"] = features["feature_idx"].astype(int)
features["layer_idx"] = features["layer_idx"].astype(int)

features["node_id"] = (
    "L" + features["layer_idx"].astype(str)
    + "-F" + features["feature_idx"].astype(str)
)

features["affected_emotion_list"] = (
    features["affected_emotions"]
    .apply(parse_affected_emotions)
)

features["affected_emotion_list"] = (
    features["affected_emotion_list"]
    .apply(lambda xs: [e for e in EMOTIONS if e in xs])
)

features["num_affected_emotions"] = (
    features["affected_emotion_list"]
    .apply(len)
)

features["is_shared"] = (
    features["num_affected_emotions"] >= 2
)

features["affected_emotions_clean"] = (
    features["affected_emotion_list"]
    .apply(lambda x: ", ".join(x))
)

# =========================
# Edge node IDs
# =========================
edges["source_feature"] = edges["source_feature"].astype(int)
edges["target_feature"] = edges["target_feature"].astype(int)

edges["source_layer"] = edges["source_layer"].astype(int)
edges["target_layer"] = edges["target_layer"].astype(int)

edges["source_node_id"] = (
    "L" + edges["source_layer"].astype(str)
    + "-F" + edges["source_feature"].astype(str)
)

edges["target_node_id"] = (
    "L" + edges["target_layer"].astype(str)
    + "-F" + edges["target_feature"].astype(str)
)

valid_nodes = set(features["node_id"])

edges = edges[
    edges["source_node_id"].isin(valid_nodes)
    & edges["target_node_id"].isin(valid_nodes)
].copy()

# =========================
# Logit columns
# =========================
LOGIT_COLS = [f"logit_{e}" for e in EMOTIONS]

existing_logit_cols = [
    c for c in LOGIT_COLS
    if c in features.columns
]

if len(existing_logit_cols) == 0:
    raise ValueError("No logit_* columns found.")

if "max_abs_logit" not in features.columns:
    features["max_abs_logit"] = (
        features[existing_logit_cols]
        .abs()
        .max(axis=1)
    )

# =========================
# Top / second emotion
# =========================
def get_top_two_logit_emotions(row):
    vals = []

    for emo in EMOTIONS:
        col = f"logit_{emo}"

        if col in row.index:
            vals.append((emo, abs(row[col])))

    vals = sorted(vals, key=lambda x: x[1], reverse=True)

    top_emo = vals[0][0]

    if len(vals) > 1:
        second_emo = vals[1][0]
    else:
        second_emo = top_emo

    return top_emo, second_emo


features[["top_logit_emotion", "second_logit_emotion"]] = (
    features.apply(
        lambda row: pd.Series(
            get_top_two_logit_emotions(row)
        ),
        axis=1,
    )
)

# =========================
# Format logit weights
# =========================
def format_logit_weights(row):
    lines = []

    for emo in EMOTIONS:
        col = f"logit_{emo}"

        if col in row.index:
            lines.append(f"{emo}: {row[col]:.4f}")

    return "<br>".join(lines)

# =========================
# Edge stats
# =========================
edges["abs_effect"] = edges["mean_effect"].abs()

if "ablated_target_mean" not in edges.columns:
    edges["ablated_target_mean"] = (
        edges["clean_target_mean"]
        - edges["mean_effect"]
    )

if "normalized_effect" in edges.columns:
    edges["drop_percent"] = (
        edges["normalized_effect"] * 100
    )
else:
    edges["drop_percent"] = (
        edges["mean_effect"]
        / (
            edges["clean_target_mean"].abs()
            + 1e-8
        )
        * 100
    )

edges["abs_drop_percent"] = (
    edges["drop_percent"].abs()
)

print(f"Nodes: {len(features)}")
print(f"Edges: {len(edges)}")
print("Shared nodes:", int(features["is_shared"].sum()))

# =========================
# Layout
# =========================
features = features.sort_values(
    ["layer_idx", "feature_idx"]
).reset_index(drop=True)

emotion_y_base = {
    emo: i * EMOTION_BAND_GAP
    for i, emo in enumerate(EMOTIONS)
}

SHARED_SECONDARY_PULL = 0.15
# 0.0 = exactly primary band
# 0.5 = halfway between primary and secondary
# 1.0 = exactly secondary band

def get_y_base(row):
    primary = row["top_logit_emotion"]
    secondary = row["second_logit_emotion"]

    primary_y = emotion_y_base[primary]

    if not row["is_shared"]:
        return primary_y

    secondary_y = emotion_y_base[secondary]

    return (
        primary_y * (1.0 - SHARED_SECONDARY_PULL)
        + secondary_y * SHARED_SECONDARY_PULL
    )


def get_layout_group(row):
    primary = row["top_logit_emotion"]

    if row["is_shared"]:
        secondary = row["second_logit_emotion"]
        return f"{primary}_shared_with_{secondary}"

    return primary


features["layout_group"] = (
    features.apply(get_layout_group, axis=1)
)

features["y_base"] = (
    features.apply(get_y_base, axis=1)
)

features["within_group_idx"] = (
    features.groupby(
        ["layout_group", "layer_idx"]
    ).cumcount()
)

features["group_count"] = (
    features.groupby(
        ["layout_group", "layer_idx"]
    )["node_id"]
    .transform("count")
)

features["y_pos"] = (
    features["y_base"]
    + features["within_group_idx"]
    - (features["group_count"] - 1) / 2
)

np.random.seed(RANDOM_SEED)

features["x_pos"] = (
    features["layer_idx"]
    + np.random.uniform(-0.06, 0.06, len(features))
)

features["y_pos"] = (
    features["y_pos"]
    + np.random.uniform(-0.06, 0.06, len(features))
)

# =========================
# Node sizes
# =========================
v = features["max_abs_logit"].to_numpy()

v = np.clip(v, 1e-6, None)

v_log = np.log(v)

v_norm = (
    (v_log - v_log.min())
    / (v_log.max() - v_log.min() + 1e-8)
)

features["node_size"] = (
    NODE_MIN_SIZE
    + v_norm * (NODE_MAX_SIZE - NODE_MIN_SIZE)
)

# =========================
# Node positions
# =========================
pos = {
    row["node_id"]: (
        row["x_pos"],
        row["y_pos"],
    )
    for _, row in features.iterrows()
}

# =========================
# Incoming / outgoing
# =========================
incoming = {
    node_id: []
    for node_id in features["node_id"]
}

outgoing = {
    node_id: []
    for node_id in features["node_id"]
}

for _, row in edges.iterrows():
    s = row["source_node_id"]
    t = row["target_node_id"]

    if s not in pos or t not in pos:
        continue

    effect = row["mean_effect"]

    clean = row["clean_target_mean"]
    ablated = row["ablated_target_mean"]

    drop = row["drop_percent"]

    outgoing[s].append(
        (
            t,
            row["target_layer"],
            effect,
            clean,
            ablated,
            drop,
        )
    )

    incoming[t].append(
        (
            s,
            row["source_layer"],
            effect,
            clean,
            ablated,
            drop,
        )
    )

for k in incoming:
    incoming[k].sort(key=lambda x: x[1])

for k in outgoing:
    outgoing[k].sort(key=lambda x: x[1])

# =========================
# Hover formatting
# =========================
def format_edges(edges_list):
    if len(edges_list) == 0:
        return "None"

    lines = []

    for other, layer, eff, clean, ablated, drop in edges_list:
        sign = "+" if eff >= 0 else ""

        lines.append(
            f"{other} (L{layer}): "
            f"clean {clean:.3f} → {ablated:.3f}, "
            f"Δ {sign}{eff:.3f}, "
            f"drop {drop:.1f}%"
        )

    return "<br>".join(lines)

# =========================
# Edge traces
# =========================
edge_traces = []

max_drop = (
    edges["abs_drop_percent"].max()
    if len(edges) > 0
    else 1.0
)

for _, row in edges.iterrows():
    s = row["source_node_id"]
    t = row["target_node_id"]

    if s not in pos or t not in pos:
        continue

    x0, y0 = pos[s]
    x1, y1 = pos[t]

    drop = abs(row["drop_percent"])
    effect = row["mean_effect"]

    width = (
        0.4
        + 4.0 * (drop / (max_drop + 1e-8))
    )

    dash = "solid" if effect >= 0 else "dot"

    color = EMOTION_COLORS.get(
        row.get("conditioning_emotion", ""),
        "#777",
    )

    edge_traces.append(
        go.Scatter(
            x=[x0, x1],
            y=[y0, y1],
            mode="lines",
            line=dict(
                width=width,
                dash=dash,
                color=color,
            ),
            opacity=EDGE_OPACITY,
            hoverinfo="text",
            text=(
                f"{s} → {t}<br>"
                f"Conditioning emotion: "
                f"{row.get('conditioning_emotion', '')}<br>"
                f"clean {row['clean_target_mean']:.3f} "
                f"→ {row['ablated_target_mean']:.3f}<br>"
                f"Δ {row['mean_effect']:.3f}, "
                f"drop {row['drop_percent']:.2f}%"
            ),
            showlegend=False,
        )
    )

# =========================
# Single-emotion nodes
# =========================
node_traces = []

for emotion in EMOTIONS:
    sub = features[
        (features["num_affected_emotions"] == 1)
        & (
            features["affected_emotion_list"]
            .apply(lambda xs: emotion in xs)
        )
    ]

    if len(sub) == 0:
        continue

    hover = []

    for _, row in sub.iterrows():
        node_id = row["node_id"]

        desc = "<br>".join(
            textwrap.wrap(
                str(row.get("description", "")),
                width=80,
            )
        )

        logit_weights = format_logit_weights(row)

        hover.append(
            f"<b>{node_id}</b><br>"
            f"Layer: {row['layer_idx']}<br>"
            f"Feature: {row['feature_idx']}<br>"
            f"<br><b>Affected emotions:</b> "
            f"{row['affected_emotions_clean']}<br>"
            f"<br><b>Shared node:</b> No<br>"
            f"<br><b>Logit weights:</b><br>"
            f"{logit_weights}<br>"
            f"<br><b>Description:</b><br>"
            f"{desc}<br>"
            f"<br><b>Incoming:</b><br>"
            f"{format_edges(incoming[node_id])}<br>"
            f"<br><b>Outgoing:</b><br>"
            f"{format_edges(outgoing[node_id])}"
        )

    node_traces.append(
        go.Scatter(
            x=sub["x_pos"],
            y=sub["y_pos"],
            mode="markers+text",
            text=sub["node_id"],
            textposition="middle center",
            textfont=dict(
                size=7,
                color="white",
            ),
            hoverinfo="text",
            hovertext=hover,
            marker=dict(
                size=sub["node_size"],
                color=EMOTION_COLORS[emotion],
                line=dict(
                    width=1,
                    color="black",
                ),
            ),
            name=emotion,
        )
    )

# =========================
# Shared nodes
# =========================
shared_nodes = features[
    features["is_shared"]
].copy()

shared_hover = []

for _, row in shared_nodes.iterrows():
    node_id = row["node_id"]

    desc = "<br>".join(
        textwrap.wrap(
            str(row.get("description", "")),
            width=80,
        )
    )

    logit_weights = format_logit_weights(row)

    shared_hover.append(
        f"<b>{node_id} (SHARED)</b><br>"
        f"Layer: {row['layer_idx']}<br>"
        f"Feature: {row['feature_idx']}<br>"
        f"<br><b>Affected emotions:</b> "
        f"{row['affected_emotions_clean']}<br>"
        f"<b># affected emotions:</b> "
        f"{row['num_affected_emotions']}<br>"
        f"<br><b>Primary emotion:</b> "
        f"{row['top_logit_emotion']}<br>"
        f"<b>Border emotion:</b> "
        f"{row['second_logit_emotion']}<br>"
        f"<br><b>Logit weights:</b><br>"
        f"{logit_weights}<br>"
        f"<br><b>Description:</b><br>"
        f"{desc}<br>"
        f"<br><b>Incoming:</b><br>"
        f"{format_edges(incoming[node_id])}<br>"
        f"<br><b>Outgoing:</b><br>"
        f"{format_edges(outgoing[node_id])}"
    )

shared_node_trace = go.Scatter(
    x=shared_nodes["x_pos"],
    y=shared_nodes["y_pos"],
    mode="markers+text",
    text=shared_nodes["node_id"],
    textposition="middle center",
    textfont=dict(
        size=7,
        color="white",
    ),
    hoverinfo="text",
    hovertext=shared_hover,
    marker=dict(
        size=shared_nodes["node_size"],
        color=shared_nodes["top_logit_emotion"]
        .map(EMOTION_COLORS),
        line=dict(
            width=3,
            color=shared_nodes["second_logit_emotion"]
            .map(EMOTION_COLORS),
        ),
    ),
    name="shared features",
    showlegend=True,
)

# =========================
# Emotion labels
# =========================
annotations = []

for emotion in EMOTIONS:
    annotations.append(
        dict(
            x=1.02,
            y=emotion_y_base[emotion],
            xref="paper",
            yref="y",
            text=f"<b>{emotion}</b>",
            showarrow=False,
            font=dict(
                size=14,
                color=EMOTION_COLORS[emotion],
            ),
            xanchor="left",
        )
    )

annotations.append(
    dict(
        x=1.02,
        y=np.mean(list(emotion_y_base.values())),
        xref="paper",
        yref="y",
        text=(
            " "
        ),
        showarrow=False,
        font=dict(
            size=11,
            color="black",
        ),
        xanchor="left",
    )
)

# =========================
# Plot
# =========================
fig = go.Figure(
    data=edge_traces
    + node_traces
    + [shared_node_trace]
)

fig.update_layout(
    title=(
        "All Emotion Features (All Edges)<br>"
        "<sup>"
        "Shared nodes use fill color for the strongest "
        "absolute logit emotion and border color for "
        "the second strongest."
        "</sup>"
    ),
    xaxis=dict(title="Layer"),
    yaxis=dict(showticklabels=False),
    width=1600,
    height=1600,
    annotations=annotations,
)

fig.show()

os.makedirs(
    os.path.dirname(OUTPUT_HTML),
    exist_ok=True,
)

fig.write_html(OUTPUT_HTML)

print("Saved to:", OUTPUT_HTML)

## gemma-3-4b-it-big

In [ ]:
from sae_emotion.emotion_entropy_utils import run_entropy_classification_pipeline


INPUT_PATH = "data/google-gemma-3-4b-it-feature-act-big/core_emotion_features_filled_logits_signed.csv"

OUTPUT_PATH = "data/google-gemma-3-4b-it-feature-act-big/core_emotion_features_filled_logits_signed_entropy.csv"

PLOT_DIR = "data/google-gemma-3-4b-it-feature-act-big/visuals"


df, threshold = run_entropy_classification_pipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    entropy_threshold=None,  # Set to 0.7204 if you want fixed threshold
    plot_dir=PLOT_DIR,
    show_plots=True,
)

print("Final entropy threshold:", threshold)

In [ ]:
import torch

from sae_emotion.logit_ablation_utils import (
    model_safe_name,
    run_topk_joint_ablation_pipeline,
    run_topk_selection_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-4b-pt"
SAE_SIZE = "big"   # "big", "medium", or "small"

SAFE_NAME = model_safe_name(MODEL_NAME)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

FEATURE_CSV_PATH = (
    f"{FEATURE_DIR}/core_emotion_features_filled_logits_signed_entropy.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

JOINT_OUT_PATH = (
    f"{FEATURE_DIR}/topk_joint_ablation_logit_drop.csv"
)

SELECTED_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_emotion_specific_topk_features.csv"
)

UNIQUE_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

# ============================================================
# 1. Run top-k joint ablation
# ============================================================

joint_df = run_topk_joint_ablation_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,
    feature_csv_path=FEATURE_CSV_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_csv_path=JOINT_OUT_PATH,
    emotions=EMOTIONS,
    top_k_list=[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20],
    cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub")),
    sample_size_per_emotion=300,
    batch_size=128,
    apply_prompt_template=False,
    use_abs_ranking=False,
    device="cuda",
    dtype=torch.bfloat16,
)

# ============================================================
# 2. Select minimal top-k core features
# ============================================================

selected_df, unique_df = run_topk_selection_pipeline(
    feature_csv_path=FEATURE_CSV_PATH,
    joint_ablation_csv_path=JOINT_OUT_PATH,
    selected_output_csv_path=SELECTED_OUT_PATH,
    unique_output_csv_path=UNIQUE_OUT_PATH,
    emotions=EMOTIONS,
    min_signed_drop=0.1,
    coverage_ratio=0.80,
)

In [ ]:
import torch
import pandas as pd

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_examples_by_emotion,
    model_safe_name,
)

from sae_emotion.emotion_graph_utils import (
    run_emotion_feature_graph_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-4b-it"
SAE_SIZE = "big"

SAFE_NAME = model_safe_name(MODEL_NAME)
FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

EMOTION_SPEC_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

OUTPUT_DIR = (
    f"{FEATURE_DIR}/emotion_feature_edges"
)

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

DEVICE = "cuda"
DTYPE = torch.bfloat16

CACHE_DIR = None

# ============================================================
# Run graph building
# ============================================================

all_edges_df = run_emotion_feature_graph_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,

    emotion_spec_csv_path=EMOTION_SPEC_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_dir=OUTPUT_DIR,
    emotions=EMOTIONS,

    text_col="input_text",
    label_col="emotion",
    sample_size_per_emotion=300,
    random_state=42,
    apply_prompt_template=False,

    device=DEVICE,
    dtype=DTYPE,
    cache_dir=CACHE_DIR,

    min_source_target_layer_gap=1,

    use_abs_effect=False,

    condition_on_target_active=False,
    target_active_threshold=0.0,
    min_target_active_count=10,
)

print(all_edges_df.shape)
print(all_edges_df.head())

## gemma-3-4b-it-small

In [ ]:
from sae_emotion.emotion_entropy_utils import run_entropy_classification_pipeline


INPUT_PATH = "data/google-gemma-3-4b-it-feature-act-small/core_emotion_features_filled_logits_signed.csv"

OUTPUT_PATH = "data/google-gemma-3-4b-it-feature-act-small/core_emotion_features_filled_logits_signed_entropy.csv"

PLOT_DIR = "data/google-gemma-3-4b-it-feature-act-small/visuals"


df, threshold = run_entropy_classification_pipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    entropy_threshold=None,  # Set to 0.7204 if you want fixed threshold
    plot_dir=PLOT_DIR,
    show_plots=True,
)

print("Final entropy threshold:", threshold)

In [ ]:
import torch

from sae_emotion.logit_ablation_utils import (
    model_safe_name,
    run_topk_joint_ablation_pipeline,
    run_topk_selection_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-4b-it"
SAE_SIZE = "small"   # "big", "medium", or "small"

SAFE_NAME = model_safe_name(MODEL_NAME)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

FEATURE_CSV_PATH = (
    f"{FEATURE_DIR}/core_emotion_features_filled_logits_signed_entropy.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

JOINT_OUT_PATH = (
    f"{FEATURE_DIR}/topk_joint_ablation_logit_drop.csv"
)

SELECTED_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_emotion_specific_topk_features.csv"
)

UNIQUE_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

# ============================================================
# 1. Run top-k joint ablation
# ============================================================

joint_df = run_topk_joint_ablation_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,
    feature_csv_path=FEATURE_CSV_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_csv_path=JOINT_OUT_PATH,
    emotions=EMOTIONS,
    top_k_list=[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20],
    # cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub")),
    sample_size_per_emotion=300,
    batch_size=128,
    apply_prompt_template=False,
    use_abs_ranking=False,
    device="cuda",
    dtype=torch.bfloat16,
)

# ============================================================
# 2. Select minimal top-k core features
# ============================================================

selected_df, unique_df = run_topk_selection_pipeline(
    feature_csv_path=FEATURE_CSV_PATH,
    joint_ablation_csv_path=JOINT_OUT_PATH,
    selected_output_csv_path=SELECTED_OUT_PATH,
    unique_output_csv_path=UNIQUE_OUT_PATH,
    emotions=EMOTIONS,
    min_signed_drop=0.1,
    coverage_ratio=0.80,
)

In [ ]:
import torch
import pandas as pd

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_examples_by_emotion,
    model_safe_name,
)

from sae_emotion.emotion_graph_utils import (
    run_emotion_feature_graph_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-4b-it"
SAE_SIZE = "small"

SAFE_NAME = model_safe_name(MODEL_NAME)
FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

EMOTION_SPEC_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

OUTPUT_DIR = (
    f"{FEATURE_DIR}/emotion_feature_edges"
)

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

DEVICE = "cuda"
DTYPE = torch.bfloat16

CACHE_DIR = None

# ============================================================
# Run graph building
# ============================================================

all_edges_df = run_emotion_feature_graph_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,

    emotion_spec_csv_path=EMOTION_SPEC_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_dir=OUTPUT_DIR,
    emotions=EMOTIONS,

    text_col="input_text",
    label_col="emotion",
    sample_size_per_emotion=300,
    random_state=42,
    apply_prompt_template=False,

    device=DEVICE,
    dtype=DTYPE,
    cache_dir=CACHE_DIR,

    min_source_target_layer_gap=1,

    use_abs_effect=False,

    condition_on_target_active=False,
    target_active_threshold=0.0,
    min_target_active_count=10,
)

print(all_edges_df.shape)
print(all_edges_df.head())

In [ ]:
import importlib
from sae_emotion import visual_utils
importlib.reload(visual_utils)

from sae_emotion.visual_utils import build_emotion_feature_graph

FEATURE_PATH = (
    "data/google-gemma-3-4b-it-feature-act-small/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

ALL_EDGE_PATH = (
    "data/google-gemma-3-4b-it-feature-act-small/"
    "emotion_feature_edges/all_affected_feature_edges.csv"
)

OUTPUT_HTML = (
    "data/google-gemma-3-4b-it-feature-act-small/"
    "visuals/core_features.html"
)

fig, features, edges = build_emotion_feature_graph(
    feature_path=FEATURE_PATH,
    edge_path=ALL_EDGE_PATH,
    output_html=OUTPUT_HTML,
    show=True,
)

## Gemma-3-12b-pt-small

In [ ]:
from sae_emotion.emotion_entropy_utils import run_entropy_classification_pipeline


INPUT_PATH = "data/google-gemma-3-12b-pt-feature-act-small/core_emotion_features_filled_logits_signed.csv"

OUTPUT_PATH = "data/google-gemma-3-12b-pt-feature-act-small/core_emotion_features_filled_logits_signed_entropy.csv"

PLOT_DIR = "data/google-gemma-3-12b-pt-feature-act-small/visuals"


df, threshold = run_entropy_classification_pipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    entropy_threshold=None,  # Set to 0.7204 if you want fixed threshold
    plot_dir=PLOT_DIR,
    show_plots=True,
)

print("Final entropy threshold:", threshold)

In [ ]:
import torch

from sae_emotion.logit_ablation_utils import (
    model_safe_name,
    run_topk_joint_ablation_pipeline,
    run_topk_selection_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-12b-pt"
SAE_SIZE = "small"   # "big", "medium", or "small"

SAFE_NAME = model_safe_name(MODEL_NAME)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

FEATURE_CSV_PATH = (
    f"{FEATURE_DIR}/core_emotion_features_filled_logits_signed_entropy.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

JOINT_OUT_PATH = (
    f"{FEATURE_DIR}/topk_joint_ablation_logit_drop.csv"
)

SELECTED_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_emotion_specific_topk_features.csv"
)

UNIQUE_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

# ============================================================
# 1. Run top-k joint ablation
# ============================================================

joint_df = run_topk_joint_ablation_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,
    feature_csv_path=FEATURE_CSV_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_csv_path=JOINT_OUT_PATH,
    emotions=EMOTIONS,
    top_k_list=[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20],
    # cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub")),
    sample_size_per_emotion=300,
    batch_size=128,
    apply_prompt_template=False,
    use_abs_ranking=False,
    device="cuda",
    dtype=torch.bfloat16,
)

# ============================================================
# 2. Select minimal top-k core features
# ============================================================

selected_df, unique_df = run_topk_selection_pipeline(
    feature_csv_path=FEATURE_CSV_PATH,
    joint_ablation_csv_path=JOINT_OUT_PATH,
    selected_output_csv_path=SELECTED_OUT_PATH,
    unique_output_csv_path=UNIQUE_OUT_PATH,
    emotions=EMOTIONS,
    min_signed_drop=0.1,
    coverage_ratio=0.80,
)

In [ ]:
import torch
import pandas as pd

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_examples_by_emotion,
    model_safe_name,
)

from sae_emotion.emotion_graph_utils import (
    run_emotion_feature_graph_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-12b-pt"
SAE_SIZE = "small"

SAFE_NAME = model_safe_name(MODEL_NAME)
FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

EMOTION_SPEC_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

OUTPUT_DIR = (
    f"{FEATURE_DIR}/emotion_feature_edges"
)

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

DEVICE = "cuda"
DTYPE = torch.bfloat16

CACHE_DIR = None

# ============================================================
# Run graph building
# ============================================================

all_edges_df = run_emotion_feature_graph_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,

    emotion_spec_csv_path=EMOTION_SPEC_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_dir=OUTPUT_DIR,
    emotions=EMOTIONS,

    text_col="input_text",
    label_col="emotion",
    sample_size_per_emotion=300,
    random_state=42,
    apply_prompt_template=False,

    device=DEVICE,
    dtype=DTYPE,
    cache_dir=CACHE_DIR,

    min_source_target_layer_gap=1,

    use_abs_effect=False,

    condition_on_target_active=False,
    target_active_threshold=0.0,
    min_target_active_count=10,
)

print(all_edges_df.shape)
print(all_edges_df.head())

In [ ]:
from sae_emotion.visual_utils import build_emotion_feature_graph

FEATURE_PATH = (
    "data/google-gemma-3-12b-pt-feature-act-small/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

ALL_EDGE_PATH = (
    "data/google-gemma-3-12b-pt-feature-act-small/"
    "emotion_feature_edges/all_affected_feature_edges.csv"
)

OUTPUT_HTML = (
    "data/google-gemma-3-12b-pt-feature-act-small/"
    "visuals/core_features.html"
)

fig, features, edges = build_emotion_feature_graph(
    feature_path=FEATURE_PATH,
    edge_path=ALL_EDGE_PATH,
    output_html=OUTPUT_HTML,
    show=True,
)

## Gemma-3-12b-pt-big

In [ ]:
from sae_emotion.emotion_entropy_utils import run_entropy_classification_pipeline


INPUT_PATH = "data/google-gemma-3-12b-pt-feature-act-big/core_emotion_features_filled_logits_signed.csv"

OUTPUT_PATH = "data/google-gemma-3-12b-pt-feature-act-big/core_emotion_features_filled_logits_signed_entropy.csv"

PLOT_DIR = "data/google-gemma-3-12b-pt-feature-act-big/visuals"


df, threshold = run_entropy_classification_pipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    entropy_threshold=None,  # Set to 0.7204 if you want fixed threshold
    plot_dir=PLOT_DIR,
    show_plots=True,
)

print("Final entropy threshold:", threshold)

In [ ]:
import torch

from sae_emotion.logit_ablation_utils import (
    model_safe_name,
    run_topk_joint_ablation_pipeline,
    run_topk_selection_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-12b-pt"
SAE_SIZE = "big"   # "big", "medium", or "small"

SAFE_NAME = model_safe_name(MODEL_NAME)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

FEATURE_CSV_PATH = (
    f"{FEATURE_DIR}/core_emotion_features_filled_logits_signed_entropy.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

JOINT_OUT_PATH = (
    f"{FEATURE_DIR}/topk_joint_ablation_logit_drop.csv"
)

SELECTED_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_emotion_specific_topk_features.csv"
)

UNIQUE_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

# ============================================================
# 1. Run top-k joint ablation
# ============================================================

joint_df = run_topk_joint_ablation_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,
    feature_csv_path=FEATURE_CSV_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_csv_path=JOINT_OUT_PATH,
    emotions=EMOTIONS,
    top_k_list=[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20],
    # cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub")),
    sample_size_per_emotion=300,
    batch_size=64,
    apply_prompt_template=False,
    use_abs_ranking=False,
    device="cuda",
    dtype=torch.bfloat16,
)

# ============================================================
# 2. Select minimal top-k core features
# ============================================================

selected_df, unique_df = run_topk_selection_pipeline(
    feature_csv_path=FEATURE_CSV_PATH,
    joint_ablation_csv_path=JOINT_OUT_PATH,
    selected_output_csv_path=SELECTED_OUT_PATH,
    unique_output_csv_path=UNIQUE_OUT_PATH,
    emotions=EMOTIONS,
    min_signed_drop=0.1,
    coverage_ratio=0.80,
)

## Gemma-3-12b-it-small

In [ ]:
from sae_emotion.emotion_entropy_utils import run_entropy_classification_pipeline


INPUT_PATH = "data/google-gemma-3-12b-it-feature-act-small/core_emotion_features_filled_logits_signed.csv"

OUTPUT_PATH = "data/google-gemma-3-12b-it-feature-act-small/core_emotion_features_filled_logits_signed_entropy.csv"

PLOT_DIR = "data/google-gemma-3-12b-it-feature-act-small/visuals"


df, threshold = run_entropy_classification_pipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    entropy_threshold=None,  # Set to 0.7204 if you want fixed threshold
    plot_dir=PLOT_DIR,
    show_plots=True,
)

print("Final entropy threshold:", threshold)

In [ ]:
import torch

from sae_emotion.logit_ablation_utils import (
    model_safe_name,
    run_topk_joint_ablation_pipeline,
    run_topk_selection_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-12b-it"
SAE_SIZE = "small"   # "big", "medium", or "small"

SAFE_NAME = model_safe_name(MODEL_NAME)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

FEATURE_CSV_PATH = (
    f"{FEATURE_DIR}/core_emotion_features_filled_logits_signed_entropy.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

JOINT_OUT_PATH = (
    f"{FEATURE_DIR}/topk_joint_ablation_logit_drop.csv"
)

SELECTED_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_emotion_specific_topk_features.csv"
)

UNIQUE_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

# ============================================================
# 1. Run top-k joint ablation
# ============================================================

joint_df = run_topk_joint_ablation_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,
    feature_csv_path=FEATURE_CSV_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_csv_path=JOINT_OUT_PATH,
    emotions=EMOTIONS,
    top_k_list=[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20],
    # cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub")),
    sample_size_per_emotion=300,
    batch_size=64,
    apply_prompt_template=False,
    use_abs_ranking=False,
    device="cuda",
    dtype=torch.bfloat16,
)

# ============================================================
# 2. Select minimal top-k core features
# ============================================================

selected_df, unique_df = run_topk_selection_pipeline(
    feature_csv_path=FEATURE_CSV_PATH,
    joint_ablation_csv_path=JOINT_OUT_PATH,
    selected_output_csv_path=SELECTED_OUT_PATH,
    unique_output_csv_path=UNIQUE_OUT_PATH,
    emotions=EMOTIONS,
    min_signed_drop=0.1,
    coverage_ratio=0.80,
)

In [ ]:
import torch
import pandas as pd

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_examples_by_emotion,
    model_safe_name,
)

from sae_emotion.emotion_graph_utils import (
    run_emotion_feature_graph_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-12b-it"
SAE_SIZE = "small"

SAFE_NAME = model_safe_name(MODEL_NAME)
FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

EMOTION_SPEC_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

OUTPUT_DIR = (
    f"{FEATURE_DIR}/emotion_feature_edges"
)

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

DEVICE = "cuda"
DTYPE = torch.bfloat16

CACHE_DIR = None

# ============================================================
# Run graph building
# ============================================================

all_edges_df = run_emotion_feature_graph_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,

    emotion_spec_csv_path=EMOTION_SPEC_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_dir=OUTPUT_DIR,
    emotions=EMOTIONS,

    text_col="input_text",
    label_col="emotion",
    sample_size_per_emotion=300,
    random_state=42,
    apply_prompt_template=False,

    device=DEVICE,
    dtype=DTYPE,
    cache_dir=CACHE_DIR,

    min_source_target_layer_gap=1,

    use_abs_effect=False,

    condition_on_target_active=False,
    target_active_threshold=0.0,
    min_target_active_count=10,
)

print(all_edges_df.shape)
print(all_edges_df.head())

In [ ]:
import importlib
from sae_emotion import visual_utils
importlib.reload(visual_utils)

from sae_emotion.visual_utils import build_emotion_feature_graph

FEATURE_PATH = (
    "data/google-gemma-3-12b-it-feature-act-small/"
    "auto_selected_unique_features_with_affected_emotions.csv"
)

ALL_EDGE_PATH = (
    "data/google-gemma-3-12b-it-feature-act-small/"
    "emotion_feature_edges/all_affected_feature_edges.csv"
)

OUTPUT_HTML = (
    "data/google-gemma-3-12b-it-feature-act-small/"
    "visuals/core_features.html"
)

fig, features, edges = build_emotion_feature_graph(
    feature_path=FEATURE_PATH,
    edge_path=ALL_EDGE_PATH,
    output_html=OUTPUT_HTML,
    show=True,
)

## Gemma-3-12b-it-big

In [ ]:
from sae_emotion.emotion_entropy_utils import run_entropy_classification_pipeline


INPUT_PATH = "data/google-gemma-3-12b-it-feature-act-big/core_emotion_features_filled_logits_signed.csv"

OUTPUT_PATH = "data/google-gemma-3-12b-it-feature-act-big/core_emotion_features_filled_logits_signed_entropy.csv"

PLOT_DIR = "data/google-gemma-3-12b-it-feature-act-big/visuals"


df, threshold = run_entropy_classification_pipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    entropy_threshold=None,  # Set to 0.7204 if you want fixed threshold
    plot_dir=PLOT_DIR,
    show_plots=True,
)

print("Final entropy threshold:", threshold)

In [ ]:
import torch

from sae_emotion.logit_ablation_utils import (
    model_safe_name,
    run_topk_joint_ablation_pipeline,
    run_topk_selection_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-12b-it"
SAE_SIZE = "big"   # "big", "medium", or "small"

SAFE_NAME = model_safe_name(MODEL_NAME)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

FEATURE_CSV_PATH = (
    f"{FEATURE_DIR}/core_emotion_features_filled_logits_signed_entropy.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

JOINT_OUT_PATH = (
    f"{FEATURE_DIR}/topk_joint_ablation_logit_drop.csv"
)

SELECTED_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_emotion_specific_topk_features.csv"
)

UNIQUE_OUT_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

# ============================================================
# 1. Run top-k joint ablation
# ============================================================

joint_df = run_topk_joint_ablation_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,
    feature_csv_path=FEATURE_CSV_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_csv_path=JOINT_OUT_PATH,
    emotions=EMOTIONS,
    top_k_list=[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20],
    # cache_dir=os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub")),
    sample_size_per_emotion=300,
    batch_size=64,
    apply_prompt_template=False,
    use_abs_ranking=False,
    device="cuda",
    dtype=torch.bfloat16,
)

# ============================================================
# 2. Select minimal top-k core features
# ============================================================

selected_df, unique_df = run_topk_selection_pipeline(
    feature_csv_path=FEATURE_CSV_PATH,
    joint_ablation_csv_path=JOINT_OUT_PATH,
    selected_output_csv_path=SELECTED_OUT_PATH,
    unique_output_csv_path=UNIQUE_OUT_PATH,
    emotions=EMOTIONS,
    min_signed_drop=0.1,
    coverage_ratio=0.80,
)

In [ ]:
import torch
import pandas as pd

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_examples_by_emotion,
    model_safe_name,
)

from sae_emotion.emotion_graph_utils import (
    run_emotion_feature_graph_pipeline,
)

# ============================================================
# Config
# ============================================================

MODEL_NAME = "google/gemma-3-12b-it"
SAE_SIZE = "big"

SAFE_NAME = model_safe_name(MODEL_NAME)
FEATURE_DIR = f"data/{SAFE_NAME}-feature-act-{SAE_SIZE}"

EMOTION_SPEC_PATH = (
    f"{FEATURE_DIR}/auto_selected_unique_features_with_affected_emotions.csv"
)

DATASET_CSV_PATH = (
    f"data/combined_zeroshot_filtered_{SAFE_NAME}.csv"
)

OUTPUT_DIR = (
    f"{FEATURE_DIR}/emotion_feature_edges"
)

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

DEVICE = "cuda"
DTYPE = torch.bfloat16

CACHE_DIR = None

# ============================================================
# Run graph building
# ============================================================

all_edges_df = run_emotion_feature_graph_pipeline(
    model_name=MODEL_NAME,
    sae_size=SAE_SIZE,

    emotion_spec_csv_path=EMOTION_SPEC_PATH,
    dataset_csv_path=DATASET_CSV_PATH,
    output_dir=OUTPUT_DIR,
    emotions=EMOTIONS,

    text_col="input_text",
    label_col="emotion",
    sample_size_per_emotion=300,
    random_state=42,
    apply_prompt_template=False,

    device=DEVICE,
    dtype=DTYPE,
    cache_dir=CACHE_DIR,

    min_source_target_layer_gap=1,

    use_abs_effect=False,

    condition_on_target_active=False,
    target_active_threshold=0.0,
    min_target_active_count=10,
)

print(all_edges_df.shape)
print(all_edges_df.head())

In [ ]:
import os

CACHE_ROOT = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_ROOT}/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_ROOT}/transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{CACHE_ROOT}/hub"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT

HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Authentication is read from HF_TOKEN or the local Hugging Face login.
# Authentication is read from HF_TOKEN or the local Hugging Face login.
import traceback
import torch

from sae_emotion.adjacent_group_path_patching_utils import run_adjacent_group_path_patching_pipeline

# ============================================================
# Config
# ============================================================

MODEL_RUNS = [

    # Exclude gemma-3-1b models.
    # ("google/gemma-3-1b-pt", None),
    # ("google/gemma-3-1b-it", None),

    # Run both big and small for Gemma-3 4B.
    #("google/gemma-3-4b-pt", "big"),
    #("google/gemma-3-4b-pt", "small"),
    ("google/gemma-3-4b-it", "big"),
    # ("google/gemma-3-4b-it", "small"),

    # Llama uses None.
    # ("meta-llama/Llama-3.1-8B", None),

    # 12B models use small, not big.
    # ("google/gemma-3-12b-pt", "small"),
    # ("google/gemma-3-12b-it", "small"),
]

EMOTIONS = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
]

SAMPLE_SIZE = 200
BATCH_SIZE = 64
DEVICE = "cuda"
DTYPE = torch.bfloat16
CACHE_DIR = os.environ.get("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface/hub"))

MIN_LAYER = None
MAX_LAYER = None
MIN_NODE_SCORE = None
EXCLUDE_LAYERS = None
MAX_FEATURES_PER_LAYER = None


def build_feature_dir(model_slug: str, sae_size):
    if sae_size is None:
        return f"data/{model_slug}-feature-act"
    return f"data/{model_slug}-feature-act-{sae_size}"


all_outputs = {}
failed_models = []

for model_name, sae_size in MODEL_RUNS:
    model_slug = model_name.replace("/", "-")
    feature_dir = build_feature_dir(model_slug, sae_size)

    pred_path = f"data/combined_zeroshot_filtered_{model_slug}.csv"

    edge_path = os.path.join(
        feature_dir,
        "emotion_feature_edges",
        "all_affected_feature_edges.csv",
    )

    output_root = os.path.join(
        feature_dir,
        "adjacent_group_route_mediation",
    )

    run_key = f"{model_slug}__sae_{sae_size if sae_size is not None else 'none'}"

    print("\n" + "=" * 90)
    print(f"Running model: {model_name}")
    print(f"Model slug:    {model_slug}")
    print(f"SAE size:      {sae_size}")
    print(f"Feature dir:   {feature_dir}")
    print(f"Pred path:     {pred_path}")
    print(f"Edge path:     {edge_path}")
    print(f"Output root:   {output_root}")
    print("=" * 90)

    if not os.path.exists(pred_path):
        print(f"[SKIP] Missing pred file: {pred_path}")
        failed_models.append((run_key, "missing pred_path"))
        continue

    if not os.path.exists(edge_path):
        print(f"[SKIP] Missing edge file: {edge_path}")
        failed_models.append((run_key, "missing edge_path"))
        continue

    try:
        outputs = run_adjacent_group_path_patching_pipeline(
            model_name=model_name,
            pred_path=pred_path,
            edge_path=edge_path,
            output_root=output_root,
            emotions=EMOTIONS,

            min_layer=MIN_LAYER,
            max_layer=MAX_LAYER,
            min_node_score=MIN_NODE_SCORE,
            exclude_layers=EXCLUDE_LAYERS,

            min_features_per_layer=1,
            max_features_per_layer=MAX_FEATURES_PER_LAYER,

            sample_size=SAMPLE_SIZE,
            random_state=42,
            batch_size=BATCH_SIZE,
            sae_size=sae_size,
            device=DEVICE,
            dtype=DTYPE,
            cache_dir=CACHE_DIR,
            resume=True,
        )

        all_outputs[run_key] = outputs

        print("\nFinished:", run_key)
        print("Selected nodes:", outputs["selected_nodes_path"])
        print("Specs:", outputs["specs_path"])
        print("Instance results:", outputs["all_instances_path"])
        print("Pair summary:", outputs["pair_summary_path"])
        print("Emotion summary:", outputs["emotion_summary_path"])

    except Exception as e:
        print(f"\n[FAILED] {run_key}")
        print(type(e).__name__, str(e))
        traceback.print_exc()
        failed_models.append((run_key, str(e)))
        continue

print("\n" + "=" * 90)
print("Sweep finished.")
print("Successful runs:", len(all_outputs))
for run_key in all_outputs:
    print("  [OK]", run_key)

print("\nFailed/skipped runs:", len(failed_models))
for run_key, reason in failed_models:
    print("  [FAILED/SKIPPED]", run_key, ":", reason)
print("=" * 90)